# 04 · Scoring robustnessSara's four pre-specified sensitivity analyses, extracted standaloneSenescence has no ground truth in human tissue, so the calls cannot be validated against a gold standard — only against internal consistency. These four are what stands in for one, which is why they gate everything downstream.Built on the `03.5` scaffold (your own standalone extraction of Ask 1) with Asks 2-4 ported from `03_senescence_burden_model_v2`.

## Configuration**Why.** Same single-key config as modules 01-03.

In [ ]:
from pathlib import Pathimport warnings; warnings.filterwarnings('ignore')from config import CFG, assert_layersimport config as CDATASET = 'psychad_aging'      # <<< the only line you changecfg = CFG.for_dataset(DATASET)cfg.echo()def why(block, question, rationale=None):    """Print the Why so it lands in executed output, not only in markdown."""    print("\n" + "=" * 78)    print(f"  {block}")    print("=" * 78)    print(f"  Q: {question}")    if rationale:        for line in rationale.split(" | "):            print(f"     {line}")    print()def gate(label, ok, detail=""):    """Fail loudly. A failed gate stops the module rather than flowing downstream."""    mark = "OK  " if ok else "FAIL"    print(f"  [{mark}] {label}{'  — ' + detail if detail else ''}")    if not ok:        raise AssertionError(f"GATE FAILED: {label}. {detail}")    return ok

## Load and build frames**Why.** Rebuilds the cell-level and donor-level frames standalone, so the robustness asks do not depend on having module 07 in memory. This is your own 03.5 extraction pattern.<sub>source: `03.5_senepy_validation_evaluation_v3.ipynb` cell 1</sub>

In [ ]:
why("Load and build frames", "Rebuilds the cell-level and donor-level frames standalone, so the robustness asks do not depend on having module 07 in memory")

In [ ]:
# ── source: 03.5_senepy_validation_evaluation_v3.ipynb cell 1 ──# =============================================================================# MODULE 03.5 — PHASE 1 · LOAD + BUILD FRAMES   [Cell 1.1]# =============================================================================# Read scored h5ad → resolve cols → standardize to canonical → build df_cells,# df_donor_ct, df_cells_model (Age + log10UMI scaled, NA-dropped).# CELLTYPE_RENAME applied HERE (same point Phase 2/3 must apply it).# Aging: Age_scaled is the primary; log10UMI_scaled is a covariate.# =============================================================================print("=" * 72); print(f"§ PHASE 1 LOAD  |  {DATASET}"); print("=" * 72)H5AD_PATH = os.path.join(PATHS["input_data"], f"all_{CONDITION_TAG}_{DATASET}_scored.h5ad")if not os.path.exists(H5AD_PATH):    raise FileNotFoundError(f"Not found: {H5AD_PATH}\n  Check CONDITION_TAG/DATASET in §1.0.")with time_step(f"§1.1 read h5ad ({fmt_size(H5AD_PATH)})"):    adata = sc.read_h5ad(H5AD_PATH)if adata.obs.columns.duplicated().any():    adata.obs = adata.obs.loc[:, ~adata.obs.columns.duplicated(keep="first")]print(f"  cells × genes : {adata.n_obs:,} × {adata.n_vars:,}")print(f"  obs columns   : {len(adata.obs.columns)}")# ── §1.2 resolve required columns ────────────────────────────────────────────obs_cols = set(adata.obs.columns)missing = [f"META['{k}']={META.get(k)!r} not in obs"           for k in ["donor","celltype","study_group","sen_score","sen_label"]           if META.get(k) is None or META.get(k) not in obs_cols]if missing:    print("  ✗ column resolution failed:"); [print("     ", m) for m in missing]    print(f"  actual obs cols ({len(obs_cols)}): {sorted(obs_cols)}")    raise KeyError(f"edit META_BY_DATASET[{DATASET!r}] in §1.0")DONOR_COL, CT_COL     = META["donor"], META["celltype"]SG_COL, SEN_SCORE_COL = META["study_group"], META["sen_score"]SEN_LABEL_COL         = META["sen_label"]AGE_COL = META.get("age") if META.get("age") in obs_cols else NoneSEX_COL = META.get("sex") if META.get("sex") in obs_cols else NoneCOH_COL = META.get("cohort") if META.get("cohort") in obs_cols else Noneprint(f"  resolved: donor={DONOR_COL!r} ct={CT_COL!r} sg={SG_COL!r} "      f"score={SEN_SCORE_COL!r} label={SEN_LABEL_COL!r}")print(f"            age={AGE_COL!r} sex={SEX_COL!r} cohort={COH_COL!r}")if IS_AGING and AGE_COL is None:    raise KeyError(f"aging mode needs an age column; META['age']={META.get('age')!r} not in obs.")# ── §1.3 standardize to canonical names + derived cols ───────────────────────ren = {}if DONOR_COL != "Donor":       ren[DONOR_COL] = "Donor"if CT_COL    != "Cell_Type":   ren[CT_COL]    = "Cell_Type"if SG_COL    != "Study_Group": ren[SG_COL]    = "Study_Group"if ren: adata.obs = adata.obs.rename(columns=ren)for std, src, default in [("Age", AGE_COL, np.nan), ("Sex", SEX_COL, "Unknown"),                          ("Cohort", COH_COL, DATASET)]:    if src and src in adata.obs.columns and src != std:        if std in adata.obs.columns: adata.obs = adata.obs.drop(columns=[std])        adata.obs = adata.obs.rename(columns={src: std})    elif std not in adata.obs.columns:        adata.obs[std] = defaultadata.obs = adata.obs.loc[:, ~adata.obs.columns.duplicated(keep="first")]adata.obs["sen_score"]    = pd.to_numeric(adata.obs[SEN_SCORE_COL], errors="coerce").valuesadata.obs["is_senescent"] = (_snc_label_str(adata.obs[SEN_LABEL_COL]) == "Senescent").astype(int)n_snc = int(adata.obs["is_senescent"].sum())print(f"  sen_score range [{adata.obs['sen_score'].min():.3f}, {adata.obs['sen_score'].max():.3f}]")print(f"  is_senescent: {n_snc:,}/{adata.n_obs:,} = {n_snc/adata.n_obs*100:.2f}% SnC")# log10 total counts (covariate; required since log10UMI is in the formula)if "log10_total_counts" not in adata.obs.columns:    umi = find_umi_col(adata.obs)    if umi is None:        raise KeyError("no UMI column found for log10_total_counts (it is a model covariate). "                       f"tried META['n_counts']={META.get('n_counts')!r}, nCount_RNA, total_counts, n_counts, nUMI")    adata.obs["log10_total_counts"] = np.log10(        pd.to_numeric(adata.obs[umi], errors="coerce").fillna(1).clip(lower=1).values)    print(f"  log10_total_counts ← log10({umi!r})")else:    print(f"  log10_total_counts (already present)")# ── §1.4 drop unannotated + CELLTYPE_RENAME (same point as Phase 2/3) ─────────_un = adata.obs["Cell_Type"].astype(str).isin(["","nan","NaN","None"]) | adata.obs["Cell_Type"].isna()if _un.any():    print(f"  dropping {int(_un.sum()):,} unannotated cells"); adata = adata[~_un,:].copy()if CELLTYPE_RENAME:    n = int(adata.obs["Cell_Type"].isin(CELLTYPE_RENAME).sum())    print(f"  CELLTYPE_RENAME: {'applied to %d cells'%n if n else 'no source labels matched (already harmonized)'}")    if n: adata.obs["Cell_Type"] = adata.obs["Cell_Type"].astype(str).replace(CELLTYPE_RENAME)# ── §1.5 df_cells ────────────────────────────────────────────────────────────core = [c for c in ["Donor","Cell_Type","is_senescent","sen_score","Study_Group",                    "Age","Sex","Cohort","log10_total_counts"] if c in adata.obs.columns]df_cells = adata.obs[core].copy()df_cells["Donor"]        = df_cells["Donor"].astype(str)df_cells["Cell_Type"]    = df_cells["Cell_Type"].astype(str)df_cells["Study_Group"]  = df_cells["Study_Group"].astype(str)df_cells["is_senescent"] = df_cells["is_senescent"].astype(int)df_cells["sen_score"]    = pd.to_numeric(df_cells["sen_score"], errors="coerce")if "Age" in df_cells:    df_cells["Age"]    = pd.to_numeric(df_cells["Age"], errors="coerce")if "Sex" in df_cells:    df_cells["Sex"]    = df_cells["Sex"].astype(str)if "Cohort" in df_cells: df_cells["Cohort"] = df_cells["Cohort"].astype(str)ACTIVE_CELL_TYPES   = df_cells["Cell_Type"].value_counts().index.tolist()CELLTYPE_ORDER_PLOT = ACTIVE_CELL_TYPES[:]   # value-count orderdef _refresh_celltype_order():    global ACTIVE_CELL_TYPES, CELLTYPE_ORDER_PLOT    ACTIVE_CELL_TYPES = df_cells["Cell_Type"].value_counts().index.tolist()    CELLTYPE_ORDER_PLOT = ([c for c in CELLTYPE_ORDER_PLOT if c in ACTIVE_CELL_TYPES] +                           [c for c in ACTIVE_CELL_TYPES if c not in CELLTYPE_ORDER_PLOT])# ── §1.6 df_donor_ct (≥ MIN_CELLS_PER_DONOR) ─────────────────────────────────agg = (df_cells.groupby(["Donor","Cell_Type"])       .agg(n_cells=("is_senescent","size"), n_snc=("is_senescent","sum"),            prop_snc=("is_senescent","mean"), mean_sen_score=("sen_score","mean"))       .reset_index())df_donor_ct = agg[agg["n_cells"] >= MIN_CELLS_PER_DONOR].reset_index(drop=True)dmeta_cols  = [c for c in ["Study_Group","Age","Sex","Cohort"] if c in df_cells.columns]dmeta       = df_cells[["Donor"]+dmeta_cols].drop_duplicates("Donor").set_index("Donor")df_donor_ct = df_donor_ct.merge(dmeta, on="Donor", how="left")print(f"  df_donor_ct: {len(df_donor_ct):,} donor×CT groups (≥{MIN_CELLS_PER_DONOR} cells)")# ── §1.8 df_cells_model (scale Age + log10UMI, NA-drop) ──────────────────────df_cells_model = df_cells.copy()if "Age" in df_cells_model:    a = df_cells_model["Age"].dropna().values    if len(a) and np.std(a, ddof=1) > 0:        df_cells_model["Age_scaled"] = (df_cells_model["Age"] - a.mean()) / a.std(ddof=1)        print(f"  Age:     μ={a.mean():.2f} σ={a.std(ddof=1):.2f} → Age_scaled")    else:        df_cells_model["Age_scaled"] = df_cells_model["Age"]        print(f"  ⚠ Age zero/empty std — Age_scaled = Age (unscaled)")if "log10_total_counts" in df_cells_model:    u = df_cells_model["log10_total_counts"].dropna().values    if len(u) and np.std(u, ddof=1) > 0:        df_cells_model["log10_total_counts_scaled"] = (            (df_cells_model["log10_total_counts"] - u.mean()) / u.std(ddof=1))        print(f"  log10UMI: μ={u.mean():.2f} σ={u.std(ddof=1):.2f} → log10_total_counts_scaled")ess = ["Donor","Cell_Type","is_senescent","sen_score","Study_Group"] + \      [c for c in ["Age_scaled","Sex","log10_total_counts_scaled"] if c in df_cells_model.columns]n0 = len(df_cells_model)df_cells_model = df_cells_model.dropna(subset=ess).reset_index(drop=True)df_cells_model["Donor"]        = df_cells_model["Donor"].astype(str)df_cells_model["Cell_Type"]    = df_cells_model["Cell_Type"].astype(str)df_cells_model["is_senescent"] = df_cells_model["is_senescent"].astype(int)print(f"  df_cells_model: {n0:,} → {len(df_cells_model):,} (dropped {n0-len(df_cells_model):,} NA)")# ── §1.9 Study_Group ordering (reference first; used in disease mode) ────────sg_in = df_cells_model["Study_Group"].dropna().unique().tolist()if REFERENCE_GROUP in sg_in:    study_group_order = [REFERENCE_GROUP] + sorted([l for l in sg_in if l != REFERENCE_GROUP])else:    study_group_order = sorted(sg_in)    if IS_DISEASE:        print(f"  ⚠ REFERENCE_GROUP {REFERENCE_GROUP!r} not in data — alphabetical")_refresh_celltype_order()# ── summary ──────────────────────────────────────────────────────────────────print(f"\n  cell types ({len(ACTIVE_CELL_TYPES)}): {ACTIVE_CELL_TYPES}")print(f"  donors: {df_cells['Donor'].nunique()}")if IS_AGING:    print(f"  Age range: [{df_cells['Age'].min():.0f}, {df_cells['Age'].max():.0f}]  (continuous primary)")else:    print(f"  Study_Group order: {study_group_order}")print("\n✓ §1.1 load complete — df_cells, df_donor_ct, df_cells_model in scope.")

## Primary specs (mode-aware)**Why.** Resolves the predictor and covariates from `mode` — Age as exposure for aging, Condition with age as covariate for disease. Same specs the primary models use, so the sensitivity analyses test the same thing the headline does.<sub>source: `03.5_senepy_validation_evaluation_v3.ipynb` cell 2</sub>

In [ ]:
why("Primary specs (mode-aware)", "Resolves the predictor and covariates from `mode` — Age as exposure for aging, Condition with age as covariate for disease")

In [ ]:
# ── source: 03.5_senepy_validation_evaluation_v3.ipynb cell 2 ──# =============================================================================# MODULE 03.5 — PHASE 1 · BUILD PRIMARY_SPECS  (mode-aware)   [Cell 1.2]# =============================================================================# aging   → CONTINUOUS Age_scaled primary (one slope per CT). Age_scaled is the#           predictor, NOT a covariate. covariates = Sex, Cohort, log10UMI_scaled.# disease → CATEGORICAL Study_Group primary, relevel(ref=REFERENCE_GROUP);#           covariates = Age_scaled, Sex, Cohort, log10UMI_scaled.# The M03 §5 wrapper's `type=="continuous"` branch handles the aging case as-is.# =============================================================================print("=" * 64); print(f"§ PHASE 1 · BUILD PRIMARY_SPECS  ({STUDY_TYPE})"); print("=" * 64)# covariate pool (log10UMI included)_cov_pool = [c for c in ["Sex", "Cohort", "log10_total_counts_scaled"]             if c in df_cells_model.columns]PRIMARY_SPECS = []if IS_AGING:    if "Age_scaled" not in df_cells_model.columns:        raise RuntimeError("aging mode needs Age_scaled in df_cells_model — check §1.1 Age scaling.")    PRIMARY_SPECS.append({        "name":            "Age",        "slug":            "age",        "label":           "Age (continuous)",        "type":            "continuous",          # ← drives wrapper's continuous branch        "var_cell":        "Age_scaled",        "reference":       None,        "level_order":     [],        "covariates_cell": list(_cov_pool),        # Sex, Cohort, log10UMI (Age_scaled is primary)    })    print(f"  [aging] continuous primary = Age_scaled")    print(f"    covariates: {_cov_pool}")else:    if len(study_group_order) < 2 or REFERENCE_GROUP not in study_group_order:        raise RuntimeError(f"disease mode: bad Study_Group order {study_group_order} / ref {REFERENCE_GROUP!r}")    non_ref = [lv for lv in study_group_order if lv != REFERENCE_GROUP]    PRIMARY_SPECS.append({        "name":            "Study_Group",        "slug":            "study_group",        "label":           "Study Group",        "type":            "categorical",        "var_cell":        "Study_Group",        "reference":       REFERENCE_GROUP,        "level_order":     non_ref,        "covariates_cell": [c for c in (["Age_scaled"] + _cov_pool) if c in df_cells_model.columns],    })    print(f"  [disease] categorical primary = Study_Group  (ref={REFERENCE_GROUP!r})")    print(f"    contrasts : {non_ref}")    print(f"    covariates: {['Age_scaled'] + _cov_pool}")print(f"\n✓ §1.2 PRIMARY_SPECS ready — {len(PRIMARY_SPECS)} spec, type="      f"{PRIMARY_SPECS[0]['type']!r}")

## Ask 1 — continuous LMM**Why.** Does the result hold WITHOUT thresholding? A binary call discards information and invites 'your effect is an artifact of dichotomizing'. Dual-engine (R glmer + Python MixedLM) so the answer does not rest on one implementation.<sub>source: `03.5_senepy_validation_evaluation_v3.ipynb` cells 3, 4, 5</sub>

In [ ]:
why("Ask 1 — continuous LMM", "Does the result hold WITHOUT thresholding? A binary call discards information and invites 'your effect is an artifact of dichotomizing'")

In [ ]:
# ── source: 03.5_senepy_validation_evaluation_v3.ipynb cell 3 ──# =============================================================================# MODULE 03.5 — PHASE 1 · §8.1a — DATA PREP + ELIGIBILITY  (no fitting yet)# =============================================================================# Builds the cell-level modeling frame for the active PRIMARY_SPEC and shows,# per cell type, what each engine WOULD fit on:#   • R glmer  → outcome = is_senescent (binary)     needs both classes present#   • Py MixedLM → outcome = sen_score (continuous)   needs nonzero variance# Eligibility uses M03 §5 thresholds: min_cells=100, min_donors=5.# Nothing is fitted here — inspect the table, then run §8.1b.# =============================================================================print("=" * 72); print(f"§8.1a — DATA PREP + ELIGIBILITY  |  {DATASET}"); print("=" * 72)SPEC = PRIMARY_SPECS[0]PRIMARY_TERM = SPEC["var_cell"]                 # aging: Age_scaled ; disease: Study_Groupprint(f"  spec      : {SPEC['label']}  (type={SPEC['type']}, primary={PRIMARY_TERM!r})")print(f"  outcomes  : R glmer → is_senescent (binary) | Py MixedLM → sen_score (continuous)")print(f"  thresholds: min_cells={GLMM_CONFIG['min_cells']}  min_donors={GLMM_CONFIG['min_donors']}")# ── assemble the columns both engines need (primary + covariates + outcomes) ──need = ["Donor", "Cell_Type", "is_senescent", "sen_score"]if SPEC["type"] == "categorical":    need.append(PRIMARY_TERM)                   # Study_Groupelse:    need.append(PRIMARY_TERM)                   # Age_scaledfor cov in SPEC["covariates_cell"]:    if cov in df_cells_model.columns and cov not in need:        need.append(cov)missing_cols = [c for c in need if c not in df_cells_model.columns]if missing_cols:    raise KeyError(f"§8.1a: columns missing from df_cells_model: {missing_cols}")df_fit = df_cells_model[need].copy()df_fit["Donor"]        = df_fit["Donor"].astype(str)df_fit["Cell_Type"]    = df_fit["Cell_Type"].astype(str)df_fit["is_senescent"] = pd.to_numeric(df_fit["is_senescent"], errors="coerce").astype("Int64")df_fit["sen_score"]    = pd.to_numeric(df_fit["sen_score"], errors="coerce")if SPEC["type"] == "categorical":    df_fit[PRIMARY_TERM] = df_fit[PRIMARY_TERM].astype(str)else:    df_fit[PRIMARY_TERM] = pd.to_numeric(df_fit[PRIMARY_TERM], errors="coerce")n0 = len(df_fit)df_fit = df_fit.dropna(subset=need).reset_index(drop=True)print(f"\n  modeling frame: {n0:,} → {len(df_fit):,} cells "      f"(dropped {n0-len(df_fit):,} NA) | {df_fit['Donor'].nunique()} donors")# ── per-CT eligibility (computed, not fitted) ────────────────────────────────rows = []for ct in CELLTYPE_ORDER_PLOT:    sub = df_fit[df_fit["Cell_Type"] == ct]    if len(sub) == 0:        continue    n_cells    = len(sub)    n_donors   = sub["Donor"].nunique()    n_snc      = int((sub["is_senescent"] == 1).sum())    pct_snc    = n_snc / n_cells * 100    score_var  = float(np.nanvar(sub["sen_score"].values))    # engine-specific eligibility    pass_cells   = n_cells  >= GLMM_CONFIG["min_cells"]    pass_donors  = n_donors >= GLMM_CONFIG["min_donors"]    binary_ok    = 0 < n_snc < n_cells            # both classes present → glmer fittable    cont_ok      = score_var > 0                  # nonzero variance → MixedLM fittable    glmer_elig   = pass_cells and pass_donors and binary_ok    mixedlm_elig = pass_cells and pass_donors and cont_ok    rows.append({        "Cell_Type": ct, "n_cells": n_cells, "n_donors": n_donors,        "n_snc": n_snc, "pct_snc": round(pct_snc, 2),        "sen_score_var": round(score_var, 5),        "glmer_eligible": glmer_elig, "mixedlm_eligible": mixedlm_elig,        "skip_reason": (""            if (glmer_elig and mixedlm_elig) else            "; ".join([r for r in [                None if pass_cells  else f"n_cells<{GLMM_CONFIG['min_cells']}",                None if pass_donors else f"n_donors<{GLMM_CONFIG['min_donors']}",                None if binary_ok   else "no SnC variance (glmer)",                None if cont_ok     else "no score variance (MixedLM)",            ] if r])),    })df_elig = pd.DataFrame(rows)# ── display ──────────────────────────────────────────────────────────────────print(f"\n  Per-CT eligibility (ordered by CELLTYPE_ORDER_PLOT):")print(f"  {'Cell_Type':<16}{'n_cells':>9}{'n_donors':>9}{'%SnC':>7}"      f"{'scoreVar':>10}{'glmer':>7}{'MixedLM':>8}  reason")print(f"  {'─'*86}")for _, r in df_elig.iterrows():    print(f"  {r['Cell_Type']:<16}{r['n_cells']:>9,}{r['n_donors']:>9}"          f"{r['pct_snc']:>6.1f}%{r['sen_score_var']:>10.5f}"          f"{'✓' if r['glmer_eligible'] else '✗':>7}"          f"{'✓' if r['mixedlm_eligible'] else '✗':>8}  {r['skip_reason']}")n_g = int(df_elig["glmer_eligible"].sum()); n_m = int(df_elig["mixedlm_eligible"].sum())print(f"\n  eligible CTs: glmer {n_g}/{len(df_elig)}  ·  MixedLM {n_m}/{len(df_elig)}")print(f"  (OVERALL pooled model is fit separately in §8.1c regardless of per-CT eligibility)")print(f"\n✓ §8.1a done — df_fit + df_elig in scope. Inspect, then run §8.1b (formulas).")

In [ ]:
# ── source: 03.5_senepy_validation_evaluation_v3.ipynb cell 4 ──# =============================================================================# MODULE 03.5 — PHASE 1 · §8.1b — RESOLVE & DISPLAY FORMULAS  (no fitting yet)# =============================================================================# ADJUSTABLE FORMULA SURFACE. Edit PRIMARY_TERM / COVARIATES / RANDOM_EFFECTS# below; both engine formulas rebuild and print. Only terms actually present in# df_fit survive — the printout shows what WILL be fit (catches silent drops).##   R glmer    : is_senescent ~ <primary> + <covariates> + <RE>   binomial(logit)#   Py MixedLM : sen_score    ~ <primary> + <covariates>,  groups=Donor   (RE intercept)# =============================================================================print("=" * 72); print(f"§8.1b — FORMULAS & CONFIG  |  {DATASET}"); print("=" * 72)# ── EDIT HERE ────────────────────────────────────────────────────────────────PRIMARY_TERM   = SPEC["var_cell"]                                   # aging: Age_scaledCOVARIATES     = ["Sex", "Cohort"]   #"log10_total_counts_scaled"   # edit freelyRANDOM_EFFECTS = "(1|Donor)"                                        # R-side RE; Py uses groups=DonorGROUP_VAR      = "Donor"# ─────────────────────────────────────────────────────────────────────────────# resolve against what's actually in df_fit (drop absent, drop single-level cat)def _term_ok(t):    if t not in df_fit.columns:        return False, "absent"    if t in {"Sex", "Cohort"} and df_fit[t].nunique() < 2:        return False, f"single-level ({df_fit[t].nunique()})"    return True, ""cov_used, cov_dropped = [], []for c in COVARIATES:    ok, why = _term_ok(c)    (cov_used if ok else cov_dropped).append(c if ok else f"{c} [{why}]")if PRIMARY_TERM not in df_fit.columns:    raise KeyError(f"§8.1b: primary {PRIMARY_TERM!r} not in df_fit.")fixed_terms = [PRIMARY_TERM] + cov_usedrhs         = " + ".join(fixed_terms)GLMM_FORMULA_R  = f"is_senescent ~ {rhs} + {RANDOM_EFFECTS}"LMM_FORMULA_PY  = f"sen_score ~ {rhs}"          # MixedLM RE handled via groups=, re_formula='1'# ── display ──────────────────────────────────────────────────────────────────print(f"\n  PRIMARY     : {PRIMARY_TERM!r}  ({SPEC['type']})")print(f"  COVARIATES  : requested {COVARIATES}")print(f"                used      {cov_used}")if cov_dropped:    print(f"                DROPPED   {cov_dropped}   ← not fit (absent/constant)")print(f"  RANDOM EFF  : {RANDOM_EFFECTS}  (group = {GROUP_VAR})")print(f"\n  ── Engine 1: R lme4::glmer  (binary burden) ──")print(f"     outcome  : is_senescent  (0/1)")print(f"     family   : binomial(link='logit')")print(f"     control  : optimizer={GLMM_CONFIG['optimizer']!r}, "      f"maxfun={GLMM_CONFIG['maxfun']}, nAGQ={GLMM_CONFIG['nAGQ']}")print(f"     FORMULA  : {GLMM_FORMULA_R}")print(f"     effect   : β = log-odds per +1 SD of {PRIMARY_TERM}  →  OR = exp(β)")print(f"\n  ── Engine 2: Python statsmodels MixedLM  (continuous score) ──")print(f"     outcome  : sen_score  (continuous)")print(f"     FORMULA  : {LMM_FORMULA_PY}    [ groups={GROUP_VAR}, re_formula='1' ]")print(f"     effect   : β = Δ sen_score per +1 SD of {PRIMARY_TERM}  (linear slope)")print(f"\n  ── How they're compared (§8.1d) ──")print(f"     Different outcomes & scales (log-odds vs score units) → compare")print(f"     DIRECTION (sign of β) and SIGNIFICANCE concordance per CT, NOT magnitude.")print(f"     Only CTs eligible for BOTH engines enter the concordance table.")print(f"\n✓ §8.1b done — GLMM_FORMULA_R, LMM_FORMULA_PY, fixed_terms, GROUP_VAR in scope.")print(f"  Next: §8.1c — fit per CT (R glmer + Py MixedLM).")

In [ ]:
# ── source: 03.5_senepy_validation_evaluation_v3.ipynb cell 5 ──# =============================================================================# MODULE 03.5 — PHASE 1 · §8.1c — FIT per CT  (R glmer + Py MixedLM)# =============================================================================# Engine 1 (R lme4::glmer)   : is_senescent ~ <fixed> + (1|Donor), binomial(logit)#                              — VERBATIM M03 §5 wrapper + isSingular flag.# Engine 2 (Py MixedLM)      : sen_score ~ <fixed>, groups=Donor, re_formula='1'.# Fits OVERALL (pooled) first as a smoke test, then per CT (CELLTYPE_ORDER_PLOT).# Stores raw rows in glmm_rows / lmm_rows for §8.1d. No FDR/plots here.# =============================================================================import statsmodels.formula.api as smfprint("=" * 72); print(f"§8.1c — FIT per CT  |  {DATASET}"); print("=" * 72)print(f"  GLMM (R)  : {GLMM_FORMULA_R}")print(f"  LMM  (Py) : {LMM_FORMULA_PY}  [groups={GROUP_VAR}]")print(f"  fixed     : {fixed_terms}")print(f"  thresholds: min_cells={GLMM_CONFIG['min_cells']} min_donors={GLMM_CONFIG['min_donors']}")RHS = " + ".join(fixed_terms)   # primary + used covariates (no RE)# ── R glmer (M03 §5 wrapper, verbatim + isSingular) ──────────────────────────def fit_glmer(data_df, label):    n_cells, n_donors = len(data_df), data_df["Donor"].nunique()    snc = float(data_df["is_senescent"].mean())    if n_cells < GLMM_CONFIG["min_cells"]:   return None, f"n_cells<{GLMM_CONFIG['min_cells']}"    if n_donors < GLMM_CONFIG["min_donors"]: return None, f"n_donors<{GLMM_CONFIG['min_donors']}"    if snc in (0.0, 1.0):                    return None, "no SnC variance"    try:        with localconverter(pandas_converter):            ro.globalenv["ct_data"] = data_df        r_code = f"""        suppressMessages(library(lme4))        tryCatch({{            model <- glmer({GLMM_FORMULA_R}, data=ct_data, family=binomial(link="logit"),                control=glmerControl(optimizer="{GLMM_CONFIG['optimizer']}",                                     optCtrl=list(maxfun={GLMM_CONFIG['maxfun']})),                nAGQ={GLMM_CONFIG['nAGQ']})            cs <- summary(model)$coefficients            list(success=TRUE, term=rownames(cs),                 est=as.numeric(cs[,"Estimate"]), se=as.numeric(cs[,"Std. Error"]),                 z=as.numeric(cs[,"z value"]), p=as.numeric(cs[,"Pr(>|z|)"]),                 re_var=as.numeric(VarCorr(model)$Donor[1]), aic=AIC(model),                 singular=isSingular(model))        }}, error=function(e) list(success=FALSE, error=as.character(e)))        """        r = ro.r(r_code)        if not r.rx2("success")[0]:            return None, f"R: {str(r.rx2('error')[0])[:60]}"        terms = list(r.rx2("term"))        if PRIMARY_TERM not in terms:            return None, f"primary term absent ({terms})"        i = terms.index(PRIMARY_TERM)        beta = float(np.array(r.rx2("est"))[i]); se = float(np.array(r.rx2("se"))[i])        return {            "Cell_Type": label, "engine": "R_glmer", "outcome": "is_senescent",            "term": PRIMARY_TERM, "N_cells": n_cells, "N_donors": n_donors,            "SnC_rate": round(snc, 4), "Beta": beta, "SE": se,            "OR": float(np.exp(beta)),            "CI_low": float(np.exp(beta - 1.96*se)), "CI_high": float(np.exp(beta + 1.96*se)),            "stat": float(np.array(r.rx2("z"))[i]), "P_value": float(np.array(r.rx2("p"))[i]),            "Donor_var": float(r.rx2("re_var")[0]), "AIC": float(r.rx2("aic")[0]),            "singular": bool(r.rx2("singular")[0]),        }, None    except Exception as e:        return None, f"py: {str(e)[:60]}"# ── Python MixedLM (sen_score, random Donor intercept) ───────────────────────def fit_mixedlm(data_df, label):    n_cells, n_donors = len(data_df), data_df["Donor"].nunique()    if n_cells < GLMM_CONFIG["min_cells"]:   return None, f"n_cells<{GLMM_CONFIG['min_cells']}"    if n_donors < GLMM_CONFIG["min_donors"]: return None, f"n_donors<{GLMM_CONFIG['min_donors']}"    sv = float(np.nanvar(data_df["sen_score"]))    if sv <= 0: return None, "no score variance"    try:        d = data_df.copy()        # z-score the response so the optimizer is well-conditioned.        # sign + significance of the Age slope are invariant to this scaling        # (we only compare direction/significance in §8.1d).        mu, sd = d["sen_score"].mean(), d["sen_score"].std(ddof=0)        d["sen_score_z"] = (d["sen_score"] - mu) / sd        formula_z = LMM_FORMULA_PY.replace("sen_score", "sen_score_z", 1)        md = smf.mixedlm(formula_z, data=d, groups=d[GROUP_VAR], re_formula="1")        with warnings.catch_warnings():            warnings.simplefilter("ignore")            mf = md.fit(method="lbfgs", reml=True)        if PRIMARY_TERM not in mf.params.index:            return None, f"primary term absent"        beta = float(mf.params[PRIMARY_TERM]); se = float(mf.bse[PRIMARY_TERM])        pval = float(mf.pvalues[PRIMARY_TERM])        if not np.isfinite(pval) or not np.isfinite(se) or se == 0:            return None, "degenerate fit (nan/0 SE)"        try:            grp_var = float(mf.cov_re.iloc[0, 0]) if mf.cov_re is not None and mf.cov_re.size else np.nan        except Exception:            grp_var = np.nan        return {            "Cell_Type": label, "engine": "Py_MixedLM", "outcome": "sen_score_z",            "term": PRIMARY_TERM, "N_cells": n_cells, "N_donors": n_donors,            "Beta": beta, "SE": se,            "CI_low": beta - 1.96*se, "CI_high": beta + 1.96*se,            "stat": float(mf.tvalues[PRIMARY_TERM]), "P_value": pval,            "Donor_var": grp_var, "converged": bool(mf.converged),        }, None    except np.linalg.LinAlgError:        return None, "singular matrix"    except Exception as e:        return None, f"py: {str(e)[:60]}"# ── helper: assemble a per-group frame with only needed cols ─────────────────def _frame(sub):    cols = ["Donor", "Cell_Type", "is_senescent", "sen_score"] + fixed_terms    d = sub[[c for c in dict.fromkeys(cols) if c in sub.columns]].copy()    d["Donor"] = d["Donor"].astype(str)    for c in fixed_terms:        if c in {"Sex", "Cohort"}: d[c] = d[c].astype(str)    return d.dropna()glmm_rows, lmm_rows = [], []# ── OVERALL (smoke test) ─────────────────────────────────────────────────────print(f"\n  ▸ OVERALL (pooled)")t = time.time()g, ge = fit_glmer(_frame(df_fit), "OVERALL")print(f"     R glmer    : {'✓ OR=%.3f p=%.2e %s'%(g['OR'],g['P_value'],sig_stars(g['P_value'])) + (' SINGULAR' if g['singular'] else '') if g else '✗ '+ge}  ({time.time()-t:.1f}s)")if g: glmm_rows.append(g)t = time.time()l, le = fit_mixedlm(_frame(df_fit), "OVERALL")print(f"     Py MixedLM : {'✓ β=%.4f p=%.2e %s'%(l['Beta'],l['P_value'],sig_stars(l['P_value'])) if l else '✗ '+le}  ({time.time()-t:.1f}s)")if l: lmm_rows.append(l)# ── per CT ───────────────────────────────────────────────────────────────────print(f"\n  ▸ per cell type")for ct in CELLTYPE_ORDER_PLOT:    sub = df_fit[df_fit["Cell_Type"] == ct]    if len(sub) == 0: continue    fr = _frame(sub)    t = time.time()    g, ge = fit_glmer(fr, ct);   tg = time.time()-t    t = time.time()    l, le = fit_mixedlm(fr, ct); tl = time.time()-t    if g: glmm_rows.append(g)    if l: lmm_rows.append(l)    g_str = (f"OR={g['OR']:.3f} p={g['P_value']:.1e}{sig_stars(g['P_value'])}" + (" SING" if g['singular'] else "")) if g else f"✗{ge}"    l_str = (f"β={l['Beta']:+.4f} p={l['P_value']:.1e}{sig_stars(l['P_value'])}") if l else f"✗{le}"    print(f"     {ct:<16} glmer[{g_str:<26}]  mixedlm[{l_str:<22}]  ({tg+tl:.1f}s)")print(f"\n  fits: glmer {len(glmm_rows)}  ·  mixedlm {len(lmm_rows)}  (incl. OVERALL)")print(f"\n✓ §8.1c done — glmm_rows, lmm_rows in scope. Next: §8.1d (FDR + tables + concordance).")

## Ask 2 — threshold sensitivity**Why.** Does the result survive a different SD cutoff? Pre-empts 'why 2SD?'. Cell 30 holds the threshold lists; edit there and re-run.<sub>source: `03_senescence_burden_model_v2.ipynb` cells 30, 32</sub>

In [ ]:
why("Ask 2 — threshold sensitivity", "Does the result survive a different SD cutoff? Pre-empts 'why 2SD?'")

In [ ]:
# ── source: 03_senescence_burden_model_v2.ipynb cell 30 ──# =============================================================================# §0 ADDENDUM — CONFIG FOR §8.2 + §8.3# =============================================================================# Drop this cell in AFTER the main §0 CONFIG cell. Adds:#   - SD_THRESHOLDS_8_2A, PCTILE_THRESHOLDS_8_2B  : threshold lists (override-able)#   - HERNANDEZ_SEGURA_UP_GENES                   : 32-gene literal panel#   - fetch_cellage()                             : cached HAGR CellAge download#   - fetch_senepy_panel()                        : reads pre-extracted SenePy#                                                    gene list from disk (NO#                                                    senepy import in omicverse)#   - load_sloan_panel()                          : sloan-based panel loader#   - ALT_PANELS_8_3                              : panel registry for §8.3#   - SCORING_METHODS_8_3, THRESHOLDS_8_3         : method + threshold registry#   - load_panel_genes()                          : dispatcher used by §8.3## IMPORTANT — SenePy panel:#   This addendum DOES NOT call `import senepy`. The senepy package only lives#   in your `senepy` conda env. Run `extract_senepy_panels.py` once in that env#   (sits at /fs/ess/PDE0075/senescence/package/) to produce gene-list files.#   This addendum reads those files from disk.## Variables you'll edit most often:#   SD_THRESHOLDS_8_2A       (default [1.5, 1.75, 2.0, 2.25, 2.5])#   PCTILE_THRESHOLDS_8_2B   (default [1.0, 2.0, 5.0])#   ALT_PANELS_8_3           (toggle which panels to include in §8.3)#   SENEPY_PACKAGE_DIR       (where extract_senepy_panels.py wrote outputs)# =============================================================================print("=" * 64)print(f"§0 ADDENDUM — config for §8.2 / §8.3  |  {DATASET}")print("=" * 64)import osimport ioimport shutilimport zipfileimport urllib.request# ─────────────────────────────────────────────────────────────────────────────# §8.2 THRESHOLD LISTS  (manual override: edit lists, re-run §8.2)# ─────────────────────────────────────────────────────────────────────────────SD_THRESHOLDS_8_2A     = [1.5, 1.75, 2.0, 2.25, 2.5]PCTILE_THRESHOLDS_8_2B = [1.0, 2.0, 5.0]SD_DEFAULT_VALUE       = 2.0   # highlighted with thicker line in plotsprint(f"\n  §8.2.a SD thresholds       : {SD_THRESHOLDS_8_2A}  (default = {SD_DEFAULT_VALUE}σ)")print(f"  §8.2.b Percentile thresholds: {PCTILE_THRESHOLDS_8_2B}")# ─────────────────────────────────────────────────────────────────────────────# Hernandez-Segura 2017 universal senescence signature (UP-regulated only)# Source: doi:10.1016/j.cub.2017.07.033, supplementary Data S2F# ─────────────────────────────────────────────────────────────────────────────HERNANDEZ_SEGURA_UP_GENES = [    "ACADVL", "ADPGK", "B4GALT7", "BCL2L2", "CCND1", "CHMP5", "DDA1", "DGKA",    "DYNLT3", "FAM214B", "GBE1", "GDNF", "KLC1", "MT-CYB", "NOL3", "P4HA2",    "PDLIM4", "PLK3", "PLXNA3", "POFUT2", "RAI14", "SCOC", "SLC10A3", "SLC16A3",    "SUSD6", "TAF13", "TMEM87B", "TOLLIP", "TSPAN13", "UFM1", "ZBTB7A", "ZNHIT1",]print(f"\n  HernandezSegura panel       : {len(HERNANDEZ_SEGURA_UP_GENES)} genes (UP-regulated)")# ─────────────────────────────────────────────────────────────────────────────# CellAge — HAGR# ─────────────────────────────────────────────────────────────────────────────CELLAGE_URL       = "https://genomics.senescence.info/cells/cellAge.zip"CELLAGE_CACHE_DIR = os.path.join(PATHS["results"], "cache")CELLAGE_CACHE     = os.path.join(CELLAGE_CACHE_DIR, "cellAge_genes.txt")CELLAGE_SUBSET    = "induce"   # one of: "induce", "inhibit", "all"def fetch_cellage(url=CELLAGE_URL, cache_path=CELLAGE_CACHE,                       subset=CELLAGE_SUBSET, force=False):    """Returns CellAge gene list. Handles .csv/.tsv inside HAGR zip.    Tolerates 'Gene symbol' / 'gene_symbol' / 'GeneSymbol' column-name styles."""    os.makedirs(os.path.dirname(cache_path), exist_ok=True)    if os.path.exists(cache_path) and not force:        with open(cache_path, "r") as f:            genes = [g.strip() for g in f.readlines() if g.strip()]        print(f"  ✓ CellAge from cache ({cache_path}): {len(genes)} genes")        return genes    try:        print(f"  ▸ Downloading CellAge from {url}")        with urllib.request.urlopen(url, timeout=30) as resp:            data = resp.read()        z = zipfile.ZipFile(io.BytesIO(data))        data_files = [n for n in z.namelist()                       if n.lower().endswith((".csv", ".tsv", ".txt"))]        if not data_files:            print(f"  ⚠ CellAge zip has no parsable file — got {z.namelist()}")            return []        fname = data_files[0]        sep = "\t" if fname.lower().endswith((".tsv", ".txt")) else ","        print(f"    Reading {fname} (sep={sep!r})")        with z.open(fname) as f:            df_ca = pd.read_csv(f, sep=sep, low_memory=False)        print(f"    Columns: {df_ca.columns.tolist()}")        def _norm(s):            return str(s).strip().lower().replace(" ", "_").replace("-", "_")        col_map = {_norm(c): c for c in df_ca.columns}        gene_keys = ["gene_symbol", "gene_name", "symbol", "gene",                       "genename", "name", "gene_names"]        gene_col = None        for k in gene_keys:            if k in col_map:                gene_col = col_map[k]                break        effect_keys = ["senescence_effect", "effect", "senescence",                         "cellage_effect", "type_of_senescence"]        effect_col = None        for k in effect_keys:            if k in col_map:                effect_col = col_map[k]                break        if gene_col is None:            print(f"  ⚠ CellAge: no recognizable gene-name column "                  f"(tried: {gene_keys})")            return []        print(f"    Using gene column: {gene_col!r}")        if effect_col:            print(f"    Using effect column: {effect_col!r}")        if subset == "all" or effect_col is None:            genes = (df_ca[gene_col].dropna().astype(str)                          .str.strip().unique().tolist())            if effect_col is None and subset != "all":                print(f"    (no effect column found; using all genes)")        else:            mask = df_ca[effect_col].astype(str).str.lower().str.contains(                "induce" if subset == "induce" else "inhibit",                na=False,            )            genes = (df_ca.loc[mask, gene_col].dropna()                          .astype(str).str.strip().unique().tolist())            print(f"    Filtered by effect='{subset}': "                  f"{int(mask.sum())}/{len(df_ca)} rows")        genes = sorted(set(g for g in genes if g and g != "nan"))        with open(cache_path, "w") as f:            for g in genes:                f.write(g + "\n")        print(f"  ✓ CellAge cached ({cache_path}): {len(genes)} genes "              f"(subset='{subset}')")        return genes    except Exception as e:        print(f"  ✗ CellAge fetch failed: {str(e)[:120]}")        return []# ─────────────────────────────────────────────────────────────────────────────# SenePy — read pre-extracted gene list from disk# (file produced by extract_senepy_panels.py running in `senepy` env)# ─────────────────────────────────────────────────────────────────────────────# Where extract_senepy_panels.py wrote its output. Files there should be named:#   senepy_brain_genes.txt#   senepy_pbmc_genes.txtSENEPY_PACKAGE_DIR = "/fs/ess/PDE0075/senescence/package"# Local cache inside Module 03's results — fast read on subsequent runsSENEPY_LOCAL_CACHE = os.path.join(CELLAGE_CACHE_DIR, f"senepy_{TISSUE}_genes.txt")def fetch_senepy_panel(tissue=None, force=False, copy_to_local=True):    """Read pre-extracted SenePy gene list from disk.    Lookup order:      1. Local Module 03 cache       (PATHS["results"]/cache/senepy_<tissue>_genes.txt)      2. Shared package directory    (/fs/ess/PDE0075/senescence/package/senepy_<tissue>_genes.txt)      3. Try to import senepy        (only works if running from the senepy env)    If found in (2), copies to (1) for faster subsequent reads.    Returns [] if file not found anywhere — §8.3 will skip SenePy in that case.    To regenerate gene lists, run extract_senepy_panels.py in the senepy env.    """    if tissue is None:        tissue = TISSUE    local_path  = SENEPY_LOCAL_CACHE if tissue == TISSUE else \                       os.path.join(CELLAGE_CACHE_DIR, f"senepy_{tissue}_genes.txt")    shared_path = os.path.join(SENEPY_PACKAGE_DIR, f"senepy_{tissue}_genes.txt")    # 1. Local Module 03 cache    if os.path.exists(local_path) and not force:        with open(local_path, "r") as f:            genes = [g.strip() for g in f.readlines() if g.strip()]        print(f"  ✓ SenePy ({tissue}) from local cache: {len(genes)} genes")        return genes    # 2. Shared package dir → copy to local cache, return    if os.path.exists(shared_path):        os.makedirs(os.path.dirname(local_path), exist_ok=True)        if copy_to_local:            shutil.copy(shared_path, local_path)            print(f"  ▸ Copying SenePy ({tissue}) from {shared_path} → local cache")        with open(shared_path, "r") as f:            genes = [g.strip() for g in f.readlines() if g.strip()]        print(f"  ✓ SenePy ({tissue}): {len(genes)} genes")        return genes    # 3. Last resort — try importing senepy (only works if user runs this in senepy env)    print(f"  ▸ SenePy ({tissue}) not found at {local_path}")    print(f"    or at {shared_path}")    print(f"    Trying live senepy import as fallback...")    try:        import senepy as sp    except ImportError:        print(f"  ⚠ senepy package not installed in this env (expected for omicverse)")        print(f"    Generate the gene list by running:")        print(f"      conda activate senepy")        print(f"      cd {SENEPY_PACKAGE_DIR}")        print(f"      python extract_senepy_panels.py")        return []    # If we reach here we're somehow in the senepy env; do the merge live    try:        TISSUE_KEYWORDS = {            "brain": ["brain", "hippocampus", "cortex", "cerebellum",                       "cerebral_cortex", "hypothalamus"],            "pbmc":  ["blood", "PBMC", "lymph", "spleen", "bone_marrow"],        }        candidates = TISSUE_KEYWORDS.get(tissue, [tissue])        hubs = sp.load_hubs(species="Human")        meta = hubs.metadata        match = None        for cand in candidates:            if cand in meta["tissue"].astype(str).unique():                match = cand                break        if match is None:            print(f"  ✗ No tissue match for {tissue!r}")            return []        filtered = meta[meta["tissue"] == match]        hubs.merge_hubs(filtered, new_name=tissue)        merged = hubs.hubs[tissue]        if isinstance(merged, dict):            genes = list(merged.keys())        else:            genes = list(merged)        genes = sorted(set(str(g) for g in genes if g))        os.makedirs(os.path.dirname(local_path), exist_ok=True)        with open(local_path, "w") as f:            for g in genes:                f.write(g + "\n")        print(f"  ✓ SenePy ({tissue}) live-fetched & cached: {len(genes)} genes")        return genes    except Exception as e:        print(f"  ✗ Live senepy fetch failed: {str(e)[:120]}")        return []# ─────────────────────────────────────────────────────────────────────────────# Sloan panel loader (reads from existing SLOAN_LISTS in your main §0)# ─────────────────────────────────────────────────────────────────────────────def load_sloan_panel(panel_name):    """Return list of genes for `panel_name` from SLOAN_FILE."""    if panel_name not in SLOAN_LISTS:        print(f"  ⚠ {panel_name!r} not in SLOAN_LISTS — available: "              f"{list(SLOAN_LISTS.keys())}")        return []    if not os.path.exists(SLOAN_FILE):        print(f"  ⚠ SLOAN_FILE not found: {SLOAN_FILE}")        return []    col_idx = SLOAN_LISTS[panel_name]["col"]    try:        df_raw = pd.read_excel(SLOAN_FILE, header=None, skiprows=1)    except Exception:        df_raw = pd.read_excel(SLOAN_FILE, header=0)    if col_idx >= df_raw.shape[1]:        print(f"  ⚠ {panel_name}: column {col_idx} out of bounds "              f"(file has {df_raw.shape[1]} cols)")        return []    genes = df_raw.iloc[:, col_idx].dropna().astype(str).str.strip().tolist()    genes = [g for g in genes if g and g != "nan"]    return genes# ─────────────────────────────────────────────────────────────────────────────# §8.3 PANEL REGISTRY# ─────────────────────────────────────────────────────────────────────────────ALT_PANELS_8_3 = {    "SenePy":          {"source": "senepy_tissue", "tissue": TISSUE},    "SenMayo":         {"source": "sloan",   "key": "SenMayo"},    "Fridman":         {"source": "sloan",   "key": "Fridman"},    "HernandezSegura": {"source": "literal", "genes": HERNANDEZ_SEGURA_UP_GENES},    "CellAge":         {"source": "fetch",   "fetcher": "cellage"},}# Lazy globals_SENEPY_GENES = NoneCELLAGE_GENES = Nonedef load_panel_genes(panel_name, adata=None):    """Return list of genes for a panel from ALT_PANELS_8_3."""    if panel_name not in ALT_PANELS_8_3:        return []    cfg    = ALT_PANELS_8_3[panel_name]    source = cfg["source"]    if source == "literal":        return list(cfg["genes"])    if source == "sloan":        return load_sloan_panel(cfg["key"])    if source == "senepy_tissue":        global _SENEPY_GENES        if _SENEPY_GENES is None:            _SENEPY_GENES = fetch_senepy_panel(tissue=cfg.get("tissue", TISSUE))        return list(_SENEPY_GENES)    if source == "fetch":        global CELLAGE_GENES        if cfg["fetcher"] == "cellage":            if CELLAGE_GENES is None:                CELLAGE_GENES = fetch_cellage()            return list(CELLAGE_GENES)        return []    return []# ─────────────────────────────────────────────────────────────────────────────# §8.3 SCORING METHODS + THRESHOLDS# ─────────────────────────────────────────────────────────────────────────────SCORING_METHODS_8_3 = [    "score_genes",     # scanpy default    "log2_median",     # Hernandez-Segura / Martins-Silva style    "aucell",          # rank-based AUC, in-house numpy implementation    "senepy_native",   # SenePy μ+2σ-within-CT applied to alternative panel]THRESHOLDS_8_3 = {    "pooled_top_2pct":   {"scope": "pooled", "top_pct": 2.0, "label": "pooled top 2%"},    "pooled_top_5pct":   {"scope": "pooled", "top_pct": 5.0, "label": "pooled top 5%"},    "per_ct_top_2pct":   {"scope": "per_ct", "top_pct": 2.0, "label": "per-CT top 2%"},    "per_ct_top_5pct":   {"scope": "per_ct", "top_pct": 5.0, "label": "per-CT top 5%"},}# ─────────────────────────────────────────────────────────────────────────────# §8.3 SUMMARY# ─────────────────────────────────────────────────────────────────────────────n_panels    = len(ALT_PANELS_8_3)n_methods   = len(SCORING_METHODS_8_3)n_thresh    = len(THRESHOLDS_8_3)n_configs   = n_panels * n_methods * n_threshprint(f"\n  §8.3 panels                 : {list(ALT_PANELS_8_3.keys())} ({n_panels})")print(f"  §8.3 scoring methods        : {SCORING_METHODS_8_3} ({n_methods})")print(f"  §8.3 thresholds             : {[v['label'] for v in THRESHOLDS_8_3.values()]} ({n_thresh})")print(f"  §8.3 total configurations   : {n_configs}  (panels × methods × thresholds)")# ─────────────────────────────────────────────────────────────────────────────# Sanity audit# ─────────────────────────────────────────────────────────────────────────────print(f"\n  Panel availability check:")for panel_name in ALT_PANELS_8_3.keys():    try:        genes = load_panel_genes(panel_name)        if not genes:            print(f"    ⚠ {panel_name:18s}  empty / not loadable")        else:            if "adata" in dir():                in_data = sum(1 for g in genes if g in adata.var_names)                print(f"    ✓ {panel_name:18s}  {len(genes):4d} genes  "                      f"({in_data} in adata)")            else:                print(f"    ✓ {panel_name:18s}  {len(genes):4d} genes")    except Exception as e:        print(f"    ✗ {panel_name:18s}  ERROR: {str(e)[:60]}")# Show M01 SenePy reference for contextif "adata" in dir() and "is_senescent" in adata.obs.columns:    n_ref = int(adata.obs["is_senescent"].astype(int).sum())    print(f"\n  M01 SenePy reference (adata.obs.is_senescent): {n_ref:,} SnC cells")print(f"\n✓ §0 addendum complete")print(f"  SenePy gene file expected at: {os.path.join(SENEPY_PACKAGE_DIR, f'senepy_{TISSUE}_genes.txt')}")print(f"  To override thresholds, edit SD_THRESHOLDS_8_2A / PCTILE_THRESHOLDS_8_2B")print(f"  To toggle panels, edit ALT_PANELS_8_3 dict")print(f"  To toggle methods, edit SCORING_METHODS_8_3 list")

In [ ]:
# ── source: 03_senescence_burden_model_v2.ipynb cell 32 ──# =============================================================================# §8.2 — THRESHOLD SENSITIVITY (Sara's robustness ask #2)# =============================================================================# Re-fits the §5 headline GLMMs across two families of threshold variants:##   §8.2.a — SD-based thresholds (within-CT μ + n*σ)#     Default: [1.5, 1.75, 2.0, 2.25, 2.5]   (override via SD_THRESHOLDS_8_2A)##   §8.2.b — Percentile-based thresholds (top X% by sen_score, per CT)#     Default: [1.0, 2.0, 5.0]               (override via PCTILE_THRESHOLDS_8_2B)## The question is NOT whether each threshold produces FDR-significant findings.# The question is whether the PATTERN (direction of effects per cell type,# rank of effect magnitudes) is preserved across thresholds.## What's varied:  threshold rule (multiple SD multipliers, multiple percentiles)# What's held:    SenePy panel, sen_score, model formulas## Outputs:#   results/s8_2_threshold_sensitivity_{slug}.csv   — per spec#   results/s8_2_threshold_sensitivity_combined.csv — all specs#   results/s8_2_threshold_pattern_summary.csv      — pattern stability check# =============================================================================# ─────────────────────────────────────────────────────────────────────────────# rpy2 — idempotent re-import# ─────────────────────────────────────────────────────────────────────────────import osos.environ.setdefault(    "R_HOME",    "/users/PAS2598/ggaitos/.conda/envs/omicverse/lib/R",)os.environ.setdefault(    "R_LIBS",    "/users/PAS2598/ggaitos/.conda/envs/omicverse/lib/R/library",)os.environ.setdefault(    "R_LIBS_USER",    "/users/PAS2598/ggaitos/.conda/envs/omicverse/lib/R/library",)import rpy2.robjects as rofrom rpy2.robjects import pandas2ri, numpy2rifrom rpy2.robjects.conversion import localconverterif "pandas_converter" not in dir() and "pandas_converter" not in globals():    pandas_converter = ro.default_converter + pandas2ri.converter + numpy2ri.converterro.r("suppressPackageStartupMessages({library(lme4); library(MASS)})")print("=" * 64)print(f"§8.2 — THRESHOLD SENSITIVITY (SD + percentile variants)  |  {DATASET}")print("=" * 64)# Defensive harmonizationif "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:    n_ren = int(df_cells_model["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())    if n_ren > 0:        print(f"  ⚠ Re-applying CT harmonization ({n_ren} cells)")        df_cells_model["Cell_Type"] = df_cells_model["Cell_Type"].replace(CELLTYPE_RENAME)_refresh_celltype_order()if "sen_score" not in df_cells_model.columns:    print(f"\n⊘ sen_score not in df_cells_model — §8.2 cannot run.")    df_thr_by_spec = {}    df_thr         = pd.DataFrame()    df_pattern     = pd.DataFrame()else:    # ─────────────────────────────────────────────────────────────────────────    # §8.2.0 — Build THRESHOLD_SCHEMES from §0 addendum config    # ─────────────────────────────────────────────────────────────────────────    # SD-based variants (§8.2.a)    THRESHOLD_SCHEMES = {}    for sd_mult in SD_THRESHOLDS_8_2A:        # key like "thr_sd_2p0", label "μ+2.0σ"        key   = f"thr_sd_{str(sd_mult).replace('.', 'p')}"        label = f"μ+{sd_mult:.2f}σ".rstrip("0").rstrip(".") + ("σ" if not f"μ+{sd_mult:.2f}σ".rstrip("0").rstrip(".").endswith("σ") else "")        # Simplify the messy label construction:        label = f"μ+{sd_mult}σ"        is_default = abs(sd_mult - SD_DEFAULT_VALUE) < 1e-9        THRESHOLD_SCHEMES[key] = {            "kind":        "sd",            "value":       sd_mult,            "label":       label,            "is_default":  is_default,        }    # Percentile-based variants (§8.2.b)    for pct in PCTILE_THRESHOLDS_8_2B:        key   = f"thr_pct_{str(pct).replace('.', 'p')}"        label = f"top {pct}%"        THRESHOLD_SCHEMES[key] = {            "kind":        "pct",            "value":       pct,            "label":       label,            "is_default":  False,        }    print(f"\n▸ Threshold schemes ({len(THRESHOLD_SCHEMES)} total):")    for key, spec in THRESHOLD_SCHEMES.items():        marker = "  (default)" if spec["is_default"] else ""        print(f"    {key:24s}  {spec['label']:14s}  kind={spec['kind']}{marker}")    # ─────────────────────────────────────────────────────────────────────────    # Compute new is_senescent_* columns per scheme    # ─────────────────────────────────────────────────────────────────────────    print(f"\n▸ Computing per-CT thresholds and binary outcome columns...")    # Per-CT μ and σ for SD-based; per-CT cutpoint for percentile-based    threshold_stats = {}    for ct in df_cells_model["Cell_Type"].unique():        sub = df_cells_model[df_cells_model["Cell_Type"] == ct]["sen_score"]        if len(sub) == 0 or sub.std() == 0:            continue        threshold_stats[ct] = {            "mu":  float(sub.mean()),            "sd":  float(sub.std()),            "n":   int(len(sub)),            "vals": sub.values,  # for percentile cutpoints        }    for thr_key, thr_spec in THRESHOLD_SCHEMES.items():        col_name = f"is_senescent_{thr_key}"        df_cells_model[col_name] = 0        for ct, stats_ct in threshold_stats.items():            ct_mask = df_cells_model["Cell_Type"] == ct            if thr_spec["kind"] == "sd":                cutpoint = stats_ct["mu"] + thr_spec["value"] * stats_ct["sd"]            else:  # "pct"                cutpoint = float(np.percentile(stats_ct["vals"],                                                    100.0 - thr_spec["value"]))            df_cells_model.loc[ct_mask, col_name] = (                df_cells_model.loc[ct_mask, "sen_score"] > cutpoint            ).astype(int)    # SnC rate summary by scheme    print(f"\n  SnC rate by threshold scheme (per CT):")    header_thrs = list(THRESHOLD_SCHEMES.keys())    header_lbls = [THRESHOLD_SCHEMES[k]["label"] for k in header_thrs]    h1 = f"  {'Cell Type':<18}"    for lbl in header_lbls:        h1 += f" {lbl:>10}"    print(h1)    print(f"  {'─' * (18 + 11 * len(header_thrs))}")    for ct in CELLTYPE_ORDER_PLOT:        if ct not in threshold_stats:            continue        line = f"  {ct:<18}"        for thr_key in header_thrs:            col = f"is_senescent_{thr_key}"            sub = df_cells_model[df_cells_model["Cell_Type"] == ct]            rate = sub[col].mean() * 100 if len(sub) > 0 else np.nan            line += f" {rate:>9.2f}%"        print(line)    # Sanity: 2σ should approximate the original is_senescent    if SD_DEFAULT_VALUE in [v["value"] for v in THRESHOLD_SCHEMES.values() if v["kind"] == "sd"]:        default_key = next(k for k, v in THRESHOLD_SCHEMES.items()                                   if v["kind"] == "sd" and v["is_default"])        agreement = (df_cells_model["is_senescent"] ==                     df_cells_model[f"is_senescent_{default_key}"]).mean()        print(f"\n  Sanity (2σ vs original is_senescent): "              f"{agreement*100:.2f}% agreement (should be ~100%)")    # ─────────────────────────────────────────────────────────────────────────    # §8.2.1 — Helper: GLMM with custom outcome column    # ─────────────────────────────────────────────────────────────────────────    def _run_glmm_threshold(data_df, spec, outcome_col, label="overall"):        """Run one logistic GLMM using a custom is_senescent column.        Returns list of result records (vs-reference contrasts only)."""        n_cells  = len(data_df)        n_donors = data_df["Donor"].nunique()        snc_rate = float(data_df[outcome_col].mean())        if n_cells < GLMM_CONFIG["min_cells"]:            return None        if n_donors < GLMM_CONFIG["min_donors"]:            return None        if snc_rate == 0 or snc_rate == 1:            return None        primary = spec["var_cell"]        cov_terms = []        for cov in spec["covariates_cell"]:            if cov not in data_df.columns:                continue            if cov in {"Sex", "Cohort"} and data_df[cov].nunique() < 2:                continue            cov_terms.append(cov)        fixed_terms  = [primary] + cov_terms        formula_full = f"{outcome_col} ~ {' + '.join(fixed_terms)} + (1|Donor)"        if spec["type"] == "continuous":            relevel_cmd = ""        else:            relevel_cmd = (                f'ct_data${primary} <- relevel('                f'factor(ct_data${primary}), ref = "{spec["reference"]}")'            )        try:            with localconverter(pandas_converter):                ro.globalenv["ct_data"] = data_df            r_code = f"""            suppressMessages(library(lme4))            {relevel_cmd}            tryCatch({{                model <- glmer(                    {formula_full},                    data    = ct_data,                    family  = binomial(link = "logit"),                    control = glmerControl(                        optimizer = "{GLMM_CONFIG['optimizer']}",                        optCtrl   = list(maxfun = {GLMM_CONFIG['maxfun']})                    ),                    nAGQ = {GLMM_CONFIG['nAGQ']}                )                coef_summary <- summary(model)$coefficients                list(success    = TRUE,                     term_names = rownames(coef_summary),                     estimates  = as.numeric(coef_summary[, "Estimate"]),                     std_errors = as.numeric(coef_summary[, "Std. Error"]),                     z_values   = as.numeric(coef_summary[, "z value"]),                     p_values   = as.numeric(coef_summary[, "Pr(>|z|)"]),                     re_var     = as.numeric(VarCorr(model)$Donor[1]),                     singular   = isSingular(model),                     aic        = AIC(model))            }}, error = function(e) {{                list(success = FALSE, error = as.character(e))            }})            """            result = ro.r(r_code)            if not result.rx2("success")[0]:                return None            coef_df = pd.DataFrame({                "term":     list(result.rx2("term_names")),                "Estimate": np.array(result.rx2("estimates")),                "SE":       np.array(result.rx2("std_errors")),                "Z":        np.array(result.rx2("z_values")),                "P_value":  np.array(result.rx2("p_values")),            })            re_var   = float(result.rx2("re_var")[0])            singular = bool(result.rx2("singular")[0])            aic      = float(result.rx2("aic")[0])            if spec["type"] == "continuous":                primary_terms = [t for t in coef_df["term"] if t == primary]            else:                primary_terms = [t for t in coef_df["term"]                                    if t.startswith(primary) and t != primary]            if not primary_terms:                return None            results = []            for term in primary_terms:                row  = coef_df[coef_df["term"] == term].iloc[0]                beta = float(row["Estimate"])                se   = float(row["SE"])                if spec["type"] == "continuous":                    contrast = primary                else:                    contrast = term.replace(primary, "").strip()                results.append({                    "Cell_Type":   label,                    "Contrast":    contrast,                    "Reference":   spec["reference"] if spec["type"] == "categorical" else None,                    "N_cells":     n_cells,                    "N_donors":    n_donors,                    "SnC_rate":    round(snc_rate, 4),                    "Beta":        round(beta, 6),                    "SE":          round(se,   6),                    "OR":          round(float(np.exp(beta)), 4),                    "CI_low":      round(float(np.exp(beta - 1.96 * se)), 4),                    "CI_high":     round(float(np.exp(beta + 1.96 * se)), 4),                    "Z":           round(float(row["Z"]), 4),                    "P_value":     round(float(row["P_value"]), 6),                    "Donor_var":   round(re_var, 4),                    "Singular":    singular,                    "AIC":         round(aic, 2),                })            return results        except Exception:            return None    # ─────────────────────────────────────────────────────────────────────────    # §8.2.2 — Iterate: spec × threshold × CT    # ─────────────────────────────────────────────────────────────────────────    df_thr_by_spec = {}    for spec in PRIMARY_SPECS:        print(f"\n{'━'*64}")        print(f"  Spec: {spec['label']}")        print(f"{'━'*64}")        if spec["var_cell"] not in df_cells_model.columns:            print(f"  ⊘ {spec['var_cell']!r} not in df_cells_model — skipping spec")            continue        spec_all_results = []        for thr_key, thr_spec in THRESHOLD_SCHEMES.items():            outcome_col = f"is_senescent_{thr_key}"            thr_label   = thr_spec["label"]            is_default  = thr_spec["is_default"]            marker      = " (default)" if is_default else ""            print(f"\n  ── Threshold: {thr_label}{marker}  [{thr_spec['kind']}] ──")            glmm_cols = ["Donor", "Cell_Type", outcome_col, spec["var_cell"]]            for cov in spec["covariates_cell"]:                if cov in df_cells_model.columns and cov not in glmm_cols:                    glmm_cols.append(cov)            df_glmm_input = df_cells_model[glmm_cols].copy().dropna()            df_glmm_input[outcome_col] = df_glmm_input[outcome_col].astype(int)            df_glmm_input["Donor"]     = df_glmm_input["Donor"].astype(str)            if spec["type"] == "categorical":                df_glmm_input[spec["var_cell"]] = df_glmm_input[spec["var_cell"]].astype(str)            for cov in spec["covariates_cell"]:                if cov in df_glmm_input.columns:                    if cov in {"Sex", "Cohort"}:                        df_glmm_input[cov] = df_glmm_input[cov].astype(str)                    else:                        df_glmm_input[cov] = pd.to_numeric(df_glmm_input[cov],                                                                errors="coerce")            df_glmm_input = df_glmm_input.dropna()            # OVERALL            overall = _run_glmm_threshold(df_glmm_input, spec, outcome_col,                                                  label="OVERALL")            if overall:                for r in overall:                    r["Threshold"]      = thr_label                    r["Threshold_key"]  = thr_key                    r["Threshold_kind"] = thr_spec["kind"]                    r["Threshold_value"] = thr_spec["value"]                    r["Is_default_thr"] = is_default                    spec_all_results.append(r)            # Per CT            for ct in CELLTYPE_ORDER_PLOT:                ct_data = df_glmm_input[df_glmm_input["Cell_Type"] == ct].copy()                ct_results = _run_glmm_threshold(ct_data, spec, outcome_col,                                                          label=ct)                if ct_results:                    for r in ct_results:                        r["Threshold"]      = thr_label                        r["Threshold_key"]  = thr_key                        r["Threshold_kind"] = thr_spec["kind"]                        r["Threshold_value"] = thr_spec["value"]                        r["Is_default_thr"] = is_default                        spec_all_results.append(r)            n_models = sum(1 for r in spec_all_results                                  if r["Threshold_key"] == thr_key)            n_sing = sum(1 for r in spec_all_results                                if r["Threshold_key"] == thr_key and r["Singular"])            print(f"    {n_models} models fit · {n_sing} singular")        if not spec_all_results:            print(f"\n  ⊘ No results for spec {spec['name']!r}")            continue        df_spec = pd.DataFrame(spec_all_results)        # FDR within (spec × threshold) — same scope as §5        df_spec["FDR"]         = np.nan        df_spec["stars"]       = "—"        df_spec["Significant"] = False        for thr_key in THRESHOLD_SCHEMES:            mask = df_spec["Threshold_key"] == thr_key            if mask.sum() == 0:                continue            df_spec.loc[mask, "FDR"] = bh_correction(                df_spec.loc[mask, "P_value"].values            ).round(6)            df_spec.loc[mask, "stars"] = (                df_spec.loc[mask, "FDR"].apply(sig_stars)            )            df_spec.loc[mask, "Significant"] = (                df_spec.loc[mask, "FDR"] < FDR_THRESHOLD            )        df_spec["Primary"] = spec["name"]        df_spec = df_spec.sort_values(            ["Threshold_kind", "Threshold_value", "Cell_Type", "Contrast"]        ).reset_index(drop=True)        # ── Display: side-by-side OR table per CT × Contrast ───────────────        # Show SD thresholds and pct thresholds separately (different scales)        for kind, kind_label in [("sd", "SD-based (μ+nσ)"),                                       ("pct", "Percentile-based (top X%)")]:            kind_keys = [k for k, v in THRESHOLD_SCHEMES.items()                            if v["kind"] == kind]            if not kind_keys:                continue            print(f"\n  ─── OR per CT × Contrast across {kind_label} thresholds ───")            kind_lbls = [THRESHOLD_SCHEMES[k]["label"] for k in kind_keys]            h = f"  {'Cell Type':<18} {'Contrast':<22}"            for lbl in kind_lbls:                h += f" {lbl:>11}"            h += f" {'%SnC range':>14}"            print(h)            print(f"  {'─' * (18 + 22 + 12 * len(kind_keys) + 16)}")            sub_kind = df_spec[df_spec["Threshold_kind"] == kind]            pivot_keys = sub_kind.groupby(                ["Cell_Type", "Contrast"], sort=False            ).first().index            for ct, contrast in pivot_keys:                sub = sub_kind[(sub_kind["Cell_Type"] == ct) &                                  (sub_kind["Contrast"] == contrast)]                ors = []                rates = []                for thr_key in kind_keys:                    row = sub[sub["Threshold_key"] == thr_key]                    if len(row) > 0:                        ors.append(float(row["OR"].iloc[0]))                        rates.append(float(row["SnC_rate"].iloc[0]) * 100)                    else:                        ors.append(np.nan)                if not rates:                    continue                rate_range = f"{min(rates):.1f}–{max(rates):.1f}%"                prefix = "► " if ct == "OVERALL" else "  "                line = f"  {prefix}{ct:<16} {contrast:<22}"                for o in ors:                    if np.isnan(o):                        line += f" {'—':>11}"                    else:                        line += f" {o:>11.3f}"                line += f" {rate_range:>14}"                print(line)        save_table(df_spec, f"s8_2_threshold_sensitivity_{spec['slug']}")        df_thr_by_spec[spec["slug"]] = df_spec    # Combined output    if df_thr_by_spec:        df_thr = pd.concat(df_thr_by_spec.values(), ignore_index=True)        save_table(df_thr, "s8_2_threshold_sensitivity_combined")    else:        df_thr = pd.DataFrame()    # ─────────────────────────────────────────────────────────────────────────    # §8.2.3 — Pattern stability summary    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}")    print(f"  §8.2 PATTERN STABILITY")    print(f"{'─'*64}")    pattern_rows = []    if len(df_thr) > 0:        for (primary, ct, contrast, kind), grp in df_thr.groupby(            ["Primary", "Cell_Type", "Contrast", "Threshold_kind"]        ):            if len(grp) < 2:                continue            ors     = grp["OR"].values            log_ors = np.log(ors)            signs   = np.sign(log_ors)            all_same_sign  = len(set(signs)) == 1            or_cv          = (grp["OR"].std() / grp["OR"].mean()                                 if grp["OR"].mean() > 0 else np.nan)            max_abs_log_or = np.max(np.abs(log_ors))            n_sig          = grp["Significant"].sum()            n_total        = len(grp)            row_dict = {                "Primary":              primary,                "Cell_Type":            ct,                "Contrast":             contrast,                "Threshold_kind":       kind,                "Direction_consistent": all_same_sign,                "OR_CV":                round(or_cv, 3) if not pd.isna(or_cv) else np.nan,                "Max_abs_log_OR":       round(max_abs_log_or, 3),                "N_thresholds_sig":     int(n_sig),                "N_thresholds_run":     int(n_total),            }            # Add per-threshold OR columns dynamically            for _, r in grp.iterrows():                row_dict[f"OR_at_{r['Threshold_key']}"] = round(float(r["OR"]), 4)            pattern_rows.append(row_dict)    df_pattern = pd.DataFrame(pattern_rows)    if len(df_pattern) > 0:        save_table(df_pattern, "s8_2_threshold_pattern_summary")        # Direction consistency        for kind, kind_label in [("sd", "SD-based"), ("pct", "Percentile-based")]:            sub = df_pattern[df_pattern["Threshold_kind"] == kind]            if len(sub) == 0:                continue            consistent = int(sub["Direction_consistent"].sum())            total      = len(sub)            pct        = consistent / total * 100 if total > 0 else 0            print(f"\n  Direction consistency [{kind_label}]:")            print(f"    {consistent}/{total} contrasts have consistent OR direction "                  f"across all thresholds  ({pct:.1f}%)")            n_all   = int((sub["N_thresholds_sig"] == sub["N_thresholds_run"]).sum())            n_some  = int((sub["N_thresholds_sig"] > 0).sum())            print(f"    {n_some} contrasts FDR-sig at ≥1 threshold")            print(f"    {n_all} contrasts FDR-sig at ALL thresholds")            # Flipped findings            flipped = sub[~sub["Direction_consistent"]]            if len(flipped) > 0:                print(f"\n  ⚠ [{kind_label}] Contrasts with INCONSISTENT direction:")                for _, row in flipped.iterrows():                    print(f"    {row['Cell_Type']:<18} {row['Contrast']:<22}  "                          f"(CV={row['OR_CV']})")            # Top stable findings            stable_strong = sub[                (sub["Direction_consistent"] == True) &                (sub["Cell_Type"] != "OVERALL")            ].sort_values("Max_abs_log_OR", ascending=False).head(8)            if len(stable_strong) > 0:                print(f"\n  Top stable [{kind_label}] effects "                      f"(consistent direction, largest |log(OR)|):")                for _, row in stable_strong.iterrows():                    print(f"    {row['Cell_Type']:<18} {row['Contrast']:<22}  "                          f"max|log(OR)|={row['Max_abs_log_OR']:.3f}  "                          f"CV={row['OR_CV']}  "                          f"sig@{row['N_thresholds_sig']}/{row['N_thresholds_run']}")    # ─────────────────────────────────────────────────────────────────────────    # §8.2.4 — Final summary    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}")    print(f"  §8.2 SUMMARY")    print(f"{'─'*64}")    n_sd_thrs  = sum(1 for v in THRESHOLD_SCHEMES.values() if v["kind"] == "sd")    n_pct_thrs = sum(1 for v in THRESHOLD_SCHEMES.values() if v["kind"] == "pct")    print(f"  Threshold count: {n_sd_thrs} SD-based + {n_pct_thrs} percentile-based "          f"= {len(THRESHOLD_SCHEMES)} total")    for spec in PRIMARY_SPECS:        slug = spec["slug"]        if slug not in df_thr_by_spec:            print(f"  {spec['name']:18s}  ⊘ no results")            continue        df_s           = df_thr_by_spec[slug]        n_unique_ct    = df_s["Cell_Type"].nunique()        n_total_models = len(df_s)        n_sig          = int(df_s["Significant"].sum())        n_sing         = int(df_s["Singular"].sum())        print(f"  {spec['name']:18s}  "              f"{n_unique_ct:2d} unique CTs · "              f"{n_total_models:3d} models (CT × threshold) · "              f"{n_sig:2d} sig · {n_sing:2d} singular")    print(f"\n✓ §8.2 Threshold sensitivity complete")    print(f"  Tables : s8_2_threshold_sensitivity_{{slug}}.csv (per spec)")    print(f"           s8_2_threshold_sensitivity_combined.csv")    print(f"           s8_2_threshold_pattern_summary.csv")    print(f"  Test   : Sara robustness #2 (threshold variation)")    print(f"  Focus  : direction & magnitude stability across "          f"{len(THRESHOLD_SCHEMES)} thresholds")    print(f"  Next   : §8.3 alternative scoring methods")

## Ask 3 — alternative scoring panels**Why.** Pre-empts 'why SenePy specifically?'. CellAge, Fridman, Hernandez-Segura and SenMayo were built on bulk fibroblast data, which is exactly why they are robustness checks and not the primary panel.<sub>source: `03_senescence_burden_model_v2.ipynb` cell 34</sub>

In [ ]:
why("Ask 3 — alternative scoring panels", "Pre-empts 'why SenePy specifically?'")

In [ ]:
# ── source: 03_senescence_burden_model_v2.ipynb cell 34 ──# =============================================================================# §8.3 — ALTERNATIVE SCORING METHODS (Sara's robustness ask #3)# =============================================================================# Tests labeling stability across 5 panels × 4 methods × 4 thresholds = 80# configurations. AGREEMENT-ONLY (no GLMM refit) to keep runtime tractable.## Outputs:#   results/s8_3_per_config.csv              — 81 rows: panel × method × threshold + SenePy native ref#   results/s8_3_pairwise_jaccard_matrix.csv — 81×81 binary-label agreement#   results/s8_3_pairwise_spearman_matrix.csv — 81×81 score-rank agreement#   results/s8_3_per_ct_jaccard.csv          — per-CT detail vs SenePy native#   results/s8_3_score_matrix.parquet        — cells × 81 configs (continuous scores)#   results/s8_3_label_matrix.parquet        — cells × 81 configs (binary labels)## DEPENDENCIES: adata_sub, df_cells_aligned (from alignment helper above)#               ALT_PANELS_8_3, SCORING_METHODS_8_3, THRESHOLDS_8_3 (from §0 addendum)# =============================================================================import timefrom scipy.sparse import issparsefrom scipy.stats import spearmanrprint("=" * 64)print(f"§8.3 — ALTERNATIVE SCORING METHODS (agreement-only)  |  {DATASET}")print("=" * 64)assert adata_sub.n_obs == len(df_cells_aligned), \    "Alignment broken — re-run §8 alignment helper"n_total       = len(df_cells_aligned)ref_celltypes = df_cells_aligned["Cell_Type"].astype(str).valuesref_labels    = df_cells_aligned["is_senescent"].astype(int).valuesref_scores    = (df_cells_aligned["sen_score"].astype(float).values                  if "sen_score" in df_cells_aligned.columns else None)n_snc_senepy = int(ref_labels.sum())print(f"  Aligned cells              : {n_total:,}")print(f"  SenePy reference SnC count : {n_snc_senepy:,} ({n_snc_senepy/n_total*100:.2f}%)")# ─────────────────────────────────────────────────────────────────────────────# §8.3.1 — Load panels via the §0 dispatcher# ─────────────────────────────────────────────────────────────────────────────print(f"\n▸ Loading panel gene lists")panels_genes = {}for panel_name in ALT_PANELS_8_3.keys():    raw_genes = load_panel_genes(panel_name)    in_data   = [g for g in raw_genes if g in adata_sub.var_names]    if len(in_data) == 0:        print(f"  ⚠ {panel_name:18s} 0 genes in adata — SKIPPING")        continue    panels_genes[panel_name] = in_data    print(f"  ✓ {panel_name:18s} {len(in_data):4d} genes (of {len(raw_genes)} raw)")if not panels_genes:    print(f"\n⊘ No panels loaded — §8.3 cannot run")    df_per_config       = pd.DataFrame()    df_pairwise_jaccard = pd.DataFrame()    df_pairwise_spear   = pd.DataFrame()else:    # ─────────────────────────────────────────────────────────────────────────    # §8.3.2 — Scoring helpers    # ─────────────────────────────────────────────────────────────────────────    def _get_expression_matrix(adata_in, layer="lognorm"):        """Return dense expression matrix (cells × genes)."""        if layer in adata_in.layers:            X = adata_in.layers[layer]        else:            X = adata_in.X        if issparse(X):            X = X.toarray()        return X    def score_method_score_genes(adata_in, gene_list, score_name):        """scanpy's sc.tl.score_genes."""        sc.tl.score_genes(            adata_in, gene_list=gene_list, score_name=score_name,            use_raw=False, random_state=SEED,        )        return adata_in.obs[score_name].values    def score_method_log2_median(adata_in, gene_list, layer="lognorm"):        """Hernandez-Segura / Martins-Silva style: per-cell mean of        (log-expression − per-gene median across cells) over the panel."""        X = _get_expression_matrix(adata_in, layer=layer)        gene_idx = [adata_in.var_names.get_loc(g) for g in gene_list                       if g in adata_in.var_names]        if not gene_idx:            return np.full(adata_in.n_obs, np.nan)        gene_expr     = X[:, gene_idx]        gene_medians  = np.median(gene_expr, axis=0)        return np.mean(gene_expr - gene_medians[None, :], axis=1)    def score_method_aucell(adata_in, gene_list, layer="lognorm",                                  rank_threshold_pct=5.0):        """In-house numpy AUCell. Per cell: rank all genes by expression,        compute the recovery curve for the panel, AUC up to the rank threshold.        rank_threshold_pct: percentile of top-ranked genes to integrate AUC over.        """        X = _get_expression_matrix(adata_in, layer=layer)  # cells × genes        n_cells, n_genes = X.shape        gene_in_panel = np.array([g in set(gene_list) for g in adata_in.var_names])        n_panel = int(gene_in_panel.sum())        if n_panel == 0:            return np.full(n_cells, np.nan)        rank_thr_n = max(1, int(np.round(n_genes * rank_threshold_pct / 100.0)))        scores = np.zeros(n_cells, dtype=np.float64)        # Vectorize over chunks to control memory        chunk_size = 500        for start in range(0, n_cells, chunk_size):            end = min(start + chunk_size, n_cells)            X_chunk = X[start:end, :]            # argsort descending → indices of top-ranked genes            # Higher expression = higher rank position            order = np.argsort(-X_chunk, axis=1, kind="stable")            # top rank_thr_n columns are the cells' top-expressed            top_genes = order[:, :rank_thr_n]            # For each cell, count cumulative panel hits along the rank            hits_mask = gene_in_panel[top_genes]  # (chunk, rank_thr_n) bool            cumulative_hits = np.cumsum(hits_mask, axis=1).astype(np.float64)            # AUC = sum of cumulative_hits / (rank_thr_n * n_panel)            # (max area = rank_thr_n × n_panel; this gives 0–1 normalization)            auc = cumulative_hits.sum(axis=1) / (rank_thr_n * n_panel)            scores[start:end] = auc        return scores    def score_method_senepy_native(adata_in, gene_list, layer="lognorm"):        """SenePy-style scoring: per-CT μ+2σ on a panel-mean score.        Score per cell = mean of (z-scored gene expression within CT) over panel.        """        # Implemented as: sum of per-gene CT-z-scores divided by sqrt(N) for each cell        # This mimics SenePy's gene-set scoring approach (panel mean of CT-internal z-scores)        X = _get_expression_matrix(adata_in, layer=layer)        gene_idx = [adata_in.var_names.get_loc(g) for g in gene_list                       if g in adata_in.var_names]        if not gene_idx:            return np.full(adata_in.n_obs, np.nan)        gene_expr = X[:, gene_idx]        cts       = adata_in.obs["Cell_Type"].astype(str).values        # Per-CT z-scores        z_expr = np.zeros_like(gene_expr, dtype=np.float64)        for ct in np.unique(cts):            ct_mask  = (cts == ct)            ct_expr  = gene_expr[ct_mask, :]            ct_mean  = ct_expr.mean(axis=0)            ct_std   = ct_expr.std(axis=0)            ct_std[ct_std == 0] = 1.0  # avoid /0            z_expr[ct_mask, :] = (ct_expr - ct_mean[None, :]) / ct_std[None, :]        return z_expr.mean(axis=1)    # ─────────────────────────────────────────────────────────────────────────    # §8.3.3 — Threshold helper (4 schemes)    # ─────────────────────────────────────────────────────────────────────────    def apply_threshold(scores, cell_types, scheme):        labels = np.zeros(len(scores), dtype=int)        top_pct = scheme["top_pct"]        if scheme["scope"] == "pooled":            if np.all(np.isnan(scores)):                return labels            thr = np.nanpercentile(scores, 100 - top_pct)            labels = (scores > thr).astype(int)        else:  # per_ct            for ct in np.unique(cell_types):                ct_mask   = (cell_types == ct)                ct_scores = scores[ct_mask]                if len(ct_scores) == 0 or np.all(np.isnan(ct_scores)):                    continue                thr = np.nanpercentile(ct_scores, 100 - top_pct)                labels[ct_mask] = (ct_scores > thr).astype(int)        return labels    def jaccard(a, b):        a = np.asarray(a, dtype=bool)        b = np.asarray(b, dtype=bool)        union = (a | b).sum()        if union == 0:            return np.nan        return float((a & b).sum()) / float(union)    # ─────────────────────────────────────────────────────────────────────────    # §8.3.4 — Generate scores for every (panel × method) combination    # ─────────────────────────────────────────────────────────────────────────    print(f"\n▸ Generating scores ({len(panels_genes)} panels × "          f"{len(SCORING_METHODS_8_3)} methods)")    t0 = time.time()    score_matrix = {}   # (panel, method) → np.array(n_total,)    for panel_name, gene_list in panels_genes.items():        print(f"\n  Panel: {panel_name} ({len(gene_list)} genes in adata)")        for method in SCORING_METHODS_8_3:            t_method = time.time()            try:                if method == "score_genes":                    sc_arr = score_method_score_genes(                        adata_sub, gene_list, f"_s83_{panel_name}_{method}"                    )                elif method == "log2_median":                    sc_arr = score_method_log2_median(adata_sub, gene_list)                elif method == "aucell":                    sc_arr = score_method_aucell(adata_sub, gene_list)                elif method == "senepy_native":                    sc_arr = score_method_senepy_native(adata_sub, gene_list)                else:                    print(f"    ⚠ Unknown method: {method}")                    continue                if np.all(np.isnan(sc_arr)):                    print(f"    ⚠ {method:14s} produced all-NaN — skipping")                    continue                score_matrix[(panel_name, method)] = sc_arr                print(f"    ✓ {method:14s} ({time.time()-t_method:.1f}s)  "                      f"score range: [{np.nanmin(sc_arr):+.3f}, {np.nanmax(sc_arr):+.3f}]")            except Exception as e:                print(f"    ✗ {method:14s} ERROR: {str(e)[:80]}")    print(f"\n  Score generation complete: {time.time()-t0:.1f}s total, "          f"{len(score_matrix)} (panel × method) score arrays")    # ─────────────────────────────────────────────────────────────────────────    # §8.3.5 — Apply 4 thresholds to each score array → label matrix    # ─────────────────────────────────────────────────────────────────────────    print(f"\n▸ Applying thresholds ({len(THRESHOLDS_8_3)} per score array)")    # config_id format: "panel|method|thr_key" — used as column key everywhere    label_matrix       = {}   # config_id → np.array(n_total,) of 0/1    score_matrix_keyed = {}   # config_id → np.array(n_total,) of continuous score    per_config_rows    = []    # First, the SenePy native reference    ref_config_id = "SenePy|native|reference"    label_matrix[ref_config_id] = ref_labels.astype(int)    if ref_scores is not None:        score_matrix_keyed[ref_config_id] = ref_scores    per_config_rows.append({        "config_id":     ref_config_id,        "Panel":         "SenePy",        "Method":        "native",        "Threshold":     "default (μ+2σ within CT)",        "Threshold_key": "reference",        "N_genes":       len(panels_genes.get("SenePy", [])),        "N_SnC":         n_snc_senepy,        "Pct_SnC":       round(n_snc_senepy / n_total * 100, 4),        "Jaccard_with_ref": 1.0,        "Spearman_with_ref": 1.0 if ref_scores is not None else np.nan,    })    # Then all alt configs    for (panel_name, method), scores in score_matrix.items():        for thr_key, thr_scheme in THRESHOLDS_8_3.items():            config_id = f"{panel_name}|{method}|{thr_key}"            labels = apply_threshold(scores, ref_celltypes, thr_scheme)            label_matrix[config_id]       = labels            score_matrix_keyed[config_id] = scores  # same scores, different threshold            n_snc   = int(labels.sum())            pct_snc = n_snc / n_total * 100            jac     = jaccard(labels, ref_labels)            sp_corr = (spearmanr(scores, ref_scores, nan_policy="omit").correlation                              if ref_scores is not None else np.nan)            per_config_rows.append({                "config_id":         config_id,                "Panel":             panel_name,                "Method":            method,                "Threshold":         thr_scheme["label"],                "Threshold_key":     thr_key,                "N_genes":           len(panels_genes[panel_name]),                "N_SnC":             n_snc,                "Pct_SnC":           round(pct_snc, 4),                "Jaccard_with_ref":  round(jac, 4),                "Spearman_with_ref": round(float(sp_corr), 4) if not pd.isna(sp_corr) else np.nan,            })    df_per_config = pd.DataFrame(per_config_rows)    save_table(df_per_config, "s8_3_per_config")    print(f"  ✓ {len(df_per_config)} configurations registered")    print(f"  ✓ Label matrix: {n_total:,} cells × {len(label_matrix)} configs")    # ─────────────────────────────────────────────────────────────────────────    # §8.3.6 — Pairwise Jaccard & Spearman (config × config)    # ─────────────────────────────────────────────────────────────────────────    print(f"\n▸ Computing pairwise matrices "          f"({len(label_matrix)}×{len(label_matrix)} = {len(label_matrix)**2:,} cells)")    config_ids = list(label_matrix.keys())    n_cfg      = len(config_ids)    # Build n_cells × n_cfg matrices once    L_mat = np.column_stack([label_matrix[cid] for cid in config_ids]).astype(bool)    S_mat = np.column_stack([        score_matrix_keyed.get(cid, np.full(n_total, np.nan)) for cid in config_ids    ])    # Pairwise Jaccard via vectorized intersection / union    print(f"  Computing Jaccard...")    inter = L_mat.T.astype(np.int64) @ L_mat.astype(np.int64)   # (n_cfg, n_cfg)    sums  = L_mat.sum(axis=0).astype(np.int64)                   # (n_cfg,)    union = sums[:, None] + sums[None, :] - inter    with np.errstate(divide="ignore", invalid="ignore"):        jaccard_mat = np.where(union > 0, inter / union, np.nan)    df_jac_mat = pd.DataFrame(jaccard_mat, index=config_ids, columns=config_ids)    save_table(df_jac_mat, "s8_3_pairwise_jaccard_matrix", index=True)    # Pairwise Spearman — use scipy on each pair    # (numpy doesn't have a vectorized rank-corr, but n_cfg ~80 is manageable)    print(f"  Computing Spearman...")    spear_mat = np.full((n_cfg, n_cfg), np.nan)    for i in range(n_cfg):        for j in range(i, n_cfg):            si = S_mat[:, i]            sj = S_mat[:, j]            valid = ~(np.isnan(si) | np.isnan(sj))            if valid.sum() < 10 or np.std(si[valid]) == 0 or np.std(sj[valid]) == 0:                continue            r = spearmanr(si[valid], sj[valid]).correlation            spear_mat[i, j] = spear_mat[j, i] = float(r)    df_spear_mat = pd.DataFrame(spear_mat, index=config_ids, columns=config_ids)    save_table(df_spear_mat, "s8_3_pairwise_spearman_matrix", index=True)    df_pairwise_jaccard = df_jac_mat    df_pairwise_spear   = df_spear_mat    # ─────────────────────────────────────────────────────────────────────────    # §8.3.7 — Per-CT Jaccard (vs SenePy native reference, within each CT)    # ─────────────────────────────────────────────────────────────────────────    print(f"\n▸ Computing per-CT Jaccard vs SenePy reference")    per_ct_rows = []    for ct in CELLTYPE_ORDER_PLOT:        ct_mask = (ref_celltypes == ct)        if ct_mask.sum() == 0:            continue        ct_ref = ref_labels[ct_mask]        for cfg_id in config_ids:            if cfg_id == ref_config_id:                continue            cfg_labels_ct = label_matrix[cfg_id][ct_mask]            jac_ct = jaccard(cfg_labels_ct, ct_ref)            panel, method, thr_key = cfg_id.split("|")            per_ct_rows.append({                "Cell_Type":     ct,                "Panel":         panel,                "Method":        method,                "Threshold_key": thr_key,                "config_id":     cfg_id,                "N_cells":       int(ct_mask.sum()),                "N_SnC_alt":     int(cfg_labels_ct.sum()),                "N_SnC_ref":     int(ct_ref.sum()),                "Jaccard":       round(float(jac_ct), 4) if not pd.isna(jac_ct) else np.nan,            })    df_per_ct = pd.DataFrame(per_ct_rows)    save_table(df_per_ct, "s8_3_per_ct_jaccard")    # ─────────────────────────────────────────────────────────────────────────    # §8.3.8 — Save raw score + label matrices (for §9.3 viz)    # ─────────────────────────────────────────────────────────────────────────    print(f"\n▸ Saving raw matrices for §9.3 visualization")    df_score_mat = pd.DataFrame(S_mat, columns=config_ids,                                       index=df_cells_aligned.index)    df_label_mat = pd.DataFrame(L_mat.astype(int), columns=config_ids,                                       index=df_cells_aligned.index)    try:        df_score_mat.to_parquet(os.path.join(PATHS["results"],                                                "s8_3_score_matrix.parquet"))        df_label_mat.to_parquet(os.path.join(PATHS["results"],                                                "s8_3_label_matrix.parquet"))        print(f"  ✓ saved score_matrix.parquet ({df_score_mat.shape})")        print(f"  ✓ saved label_matrix.parquet ({df_label_mat.shape})")    except Exception as e:        # parquet sometimes fails on weird columns; fall back to csv        print(f"  ⚠ parquet failed ({e}); falling back to csv")        df_score_mat.to_csv(os.path.join(PATHS["results"], "s8_3_score_matrix.csv"))        df_label_mat.to_csv(os.path.join(PATHS["results"], "s8_3_label_matrix.csv"))    # ─────────────────────────────────────────────────────────────────────────    # §8.3.9 — Console summary    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}")    print(f"  §8.3 SUMMARY  (Jaccard with SenePy reference)")    print(f"{'─'*64}")    # Median Jaccard with reference, by panel    ref_jac_by_panel = (        df_per_config[df_per_config["Panel"] != "SenePy"]        .groupby("Panel")        .agg(            Jaccard_min=("Jaccard_with_ref", "min"),            Jaccard_median=("Jaccard_with_ref", "median"),            Jaccard_max=("Jaccard_with_ref", "max"),            N_configs=("Jaccard_with_ref", "count"),        ).round(3).reset_index()    )    print(f"\n  By panel (vs SenePy native):")    print(ref_jac_by_panel.to_string(index=False))    # Median Jaccard, by method    ref_jac_by_method = (        df_per_config[df_per_config["Method"] != "native"]        .groupby("Method")        .agg(            Jaccard_median=("Jaccard_with_ref", "median"),            Jaccard_max=("Jaccard_with_ref", "max"),            N_configs=("Jaccard_with_ref", "count"),        ).round(3).reset_index()    )    print(f"\n  By scoring method (vs SenePy native):")    print(ref_jac_by_method.to_string(index=False))    # Top 5 most-similar configs to SenePy    top5 = (        df_per_config[df_per_config["config_id"] != ref_config_id]        .nlargest(5, "Jaccard_with_ref")        [["Panel", "Method", "Threshold", "Pct_SnC",          "Jaccard_with_ref", "Spearman_with_ref"]]    )    print(f"\n  Top-5 configs most similar to SenePy native:")    print(top5.to_string(index=False))    print(f"\n{'─'*64}")    print(f"  §8.3 COMPLETE")    print(f"{'─'*64}")    print(f"  Tables :")    print(f"    s8_3_per_config.csv               ({len(df_per_config)} rows)")    print(f"    s8_3_pairwise_jaccard_matrix.csv  ({n_cfg}×{n_cfg})")    print(f"    s8_3_pairwise_spearman_matrix.csv ({n_cfg}×{n_cfg})")    print(f"    s8_3_per_ct_jaccard.csv           ({len(df_per_ct)} rows)")    print(f"    s8_3_score_matrix.parquet         ({n_total}×{n_cfg})")    print(f"    s8_3_label_matrix.parquet         ({n_total}×{n_cfg})")    print(f"  Test  : Sara robustness #3 (panel × scoring × threshold)")    print(f"  Focus : Jaccard with SenePy reference + 80×80 pairwise matrices")    print(f"  Next  : §8.4 negative control")

## Ask 4 — negative control**Why.** Does SenePy beat what NON-senescence genes produce? The random panels must be drawn EXCLUDING SenePy genes — a random panel that happens to contain senescence genes is not a null. This is the closest thing available to a ground truth, since senescence has none in human tissue.<sub>source: `03_senescence_burden_model_v2.ipynb` cell 35</sub>

In [ ]:
why("Ask 4 — negative control", "Does SenePy beat what NON-senescence genes produce? The random panels must be drawn EXCLUDING SenePy genes — a random panel that happens to contain senescence genes is not a null")

In [ ]:
# ── source: 03_senescence_burden_model_v2.ipynb cell 35 ──# =============================================================================# §8.4 — NEGATIVE CONTROL (Sara's robustness ask #4)# =============================================================================# Tests whether the SenePy pipeline produces significant pathology effects# when fed NON-senescence genes. If yes → pipeline is biased; if no → pipeline# depends on senescence-specific gene content.## Three control conditions × two scoring methods:#   1. HVGs (deterministic, top N)#   2. Random genes (100 replicates, sampled outside any known panel)#   3. Random within HVGs (100 replicates, sampled from HVG pool only)## Outputs:#   results/s8_4_negctrl_label_summary.csv#   results/s8_4_negctrl_glmm_results.csv#   results/s8_4_negctrl_replicate_summary.csv## DEPENDENCIES: adata_sub, df_cells_aligned (from alignment helper)# =============================================================================import timefrom scipy.sparse import issparse# rpy2 idempotent setupimport osos.environ.setdefault("R_HOME", "/users/PAS2598/ggaitos/.conda/envs/omicverse/lib/R")os.environ.setdefault("R_LIBS", "/users/PAS2598/ggaitos/.conda/envs/omicverse/lib/R/library")os.environ.setdefault("R_LIBS_USER", "/users/PAS2598/ggaitos/.conda/envs/omicverse/lib/R/library")import rpy2.robjects as rofrom rpy2.robjects import pandas2ri, numpy2rifrom rpy2.robjects.conversion import localconverterif "pandas_converter" not in dir() and "pandas_converter" not in globals():    pandas_converter = ro.default_converter + pandas2ri.converter + numpy2ri.converterro.r("suppressPackageStartupMessages({library(lme4); library(MASS)})")print("=" * 64)print(f"§8.4 — NEGATIVE CONTROL  |  {DATASET}")print("=" * 64)# Defensive harmonizationif "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:    n_ren = int(df_cells_aligned["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())    if n_ren > 0:        print(f"  ⚠ Re-applying CT harmonization ({n_ren} cells)")        df_cells_aligned["Cell_Type"] = df_cells_aligned["Cell_Type"].replace(CELLTYPE_RENAME)_refresh_celltype_order()# ─────────────────────────────────────────────────────────────────────────────# §8.4.0 — Configuration# ─────────────────────────────────────────────────────────────────────────────N_REPLICATES = 100SEED_8_4 = SEEDSD_MULTIPLIER_8_4 = 2.0CONTROL_SCORING_METHODS = ["score_genes", "log2_median"]OUTCOME_COL_8_4 = "neg_label"print(f"\n▸ Configuration")print(f"  Replicates (random)  : {N_REPLICATES}")print(f"  Threshold rule       : μ+{SD_MULTIPLIER_8_4}σ within CT")print(f"  Scoring methods      : {CONTROL_SCORING_METHODS}")assert adata_sub.n_obs == len(df_cells_aligned), \    "Alignment broken — re-run §8 alignment helper"n_total       = len(df_cells_aligned)ref_celltypes = df_cells_aligned["Cell_Type"].astype(str).values# ─────────────────────────────────────────────────────────────────────────────# §8.4.1 — Build exclusion set + target panel size (matched to SenMayo)# ─────────────────────────────────────────────────────────────────────────────panel_exclude = set()for panel_name, panel_cfg in SLOAN_LISTS.items():    col_idx = panel_cfg["col"]    df_sloan_8_4 = pd.read_excel(SLOAN_FILE, header=None, skiprows=1)    genes = df_sloan_8_4.iloc[:, col_idx].dropna().astype(str).tolist()    panel_exclude.update(genes)# Also exclude the §8.3 alt panels (Hernandez-Segura, CellAge, SenePy gene list)for panel_name in ALT_PANELS_8_3.keys():    try:        gs = load_panel_genes(panel_name)        panel_exclude.update(gs)    except Exception:        passprint(f"  Excluding from control panels: {len(panel_exclude)} senescence-related genes")# Target panel size — match SenMayosloan_senmayo_genes = (    df_sloan_8_4.iloc[:, SLOAN_LISTS["SenMayo"]["col"]]    .dropna().astype(str).tolist())N_PANEL_SIZE = len([g for g in sloan_senmayo_genes if g in adata_sub.var_names])print(f"  Target panel size (SenMayo in-data): {N_PANEL_SIZE} genes")# ─────────────────────────────────────────────────────────────────────────────# §8.4.2 — Candidate gene pools# ─────────────────────────────────────────────────────────────────────────────all_genes = set(adata_sub.var_names)candidate_random = sorted(all_genes - panel_exclude)if "highly_variable_rank" in adata_sub.var.columns:    hvg_ranked    = adata_sub.var["highly_variable_rank"].dropna().sort_values()    hvg_genes_all = hvg_ranked.index.tolist()elif "highly_variable" in adata_sub.var.columns:    hvg_genes_all = adata_sub.var.index[adata_sub.var["highly_variable"]].tolist()elif "variances_norm" in adata_sub.var.columns:    hvg_genes_all = (adata_sub.var["variances_norm"]                          .sort_values(ascending=False).index.tolist())else:    hvg_genes_all = []candidate_hvg = [g for g in hvg_genes_all if g not in panel_exclude]print(f"\n  Random pool (excl. panels): {len(candidate_random):,}")print(f"  HVG pool (excl. panels)   : {len(candidate_hvg):,}")assert len(candidate_random) >= N_PANEL_SIZEassert len(candidate_hvg)    >= N_PANEL_SIZE# ─────────────────────────────────────────────────────────────────────────────# §8.4.3 — Scoring helpers# ─────────────────────────────────────────────────────────────────────────────def _score_genes_84(adata_in, gene_list, score_name):    sc.tl.score_genes(adata_in, gene_list=gene_list, score_name=score_name,                       use_raw=False, random_state=SEED)    return adata_in.obs[score_name].valuesdef _log2_median_84(adata_in, gene_list, layer="lognorm"):    if layer in adata_in.layers:        X = adata_in.layers[layer]    else:        X = adata_in.X    if issparse(X):        X = X.toarray()    gene_idx = [adata_in.var_names.get_loc(g) for g in gene_list                   if g in adata_in.var_names]    if not gene_idx:        return np.full(adata_in.n_obs, np.nan)    gene_expr    = X[:, gene_idx]    gene_medians = np.median(gene_expr, axis=0)    return np.mean(gene_expr - gene_medians[None, :], axis=1)def _threshold_sd_per_ct(scores, cell_types, sd_mult=2.0):    labels = np.zeros(len(scores), dtype=int)    for ct in np.unique(cell_types):        ct_mask   = (cell_types == ct)        ct_scores = scores[ct_mask]        if len(ct_scores) == 0 or np.all(np.isnan(ct_scores)):            continue        mu = np.nanmean(ct_scores)        sd = np.nanstd(ct_scores)        if sd == 0:            continue        labels[ct_mask] = (ct_scores > mu + sd_mult * sd).astype(int)    return labels# ─────────────────────────────────────────────────────────────────────────────# §8.4.4 — GLMM helper (R-safe outcome via backticks)# ─────────────────────────────────────────────────────────────────────────────def _run_glmm_negctrl(data_df, spec, outcome_col, label="overall"):    n_cells  = len(data_df)    n_donors = data_df["Donor"].nunique()    snc_rate = float(data_df[outcome_col].mean())    if n_cells < GLMM_CONFIG["min_cells"] or n_donors < GLMM_CONFIG["min_donors"]:        return None    if snc_rate == 0 or snc_rate == 1:        return None    primary = spec["var_cell"]    cov_terms = []    for cov in spec["covariates_cell"]:        if cov not in data_df.columns:            continue        if cov in {"Sex", "Cohort"} and data_df[cov].nunique() < 2:            continue        cov_terms.append(cov)    fixed_terms  = [primary] + cov_terms    formula_full = f"`{outcome_col}` ~ {' + '.join(fixed_terms)} + (1|Donor)"    relevel_cmd = ("" if spec["type"] == "continuous"                   else f'ct_data${primary} <- relevel(factor(ct_data${primary}), '                        f'ref = "{spec["reference"]}")')    try:        with localconverter(pandas_converter):            ro.globalenv["ct_data"] = data_df        r_code = f"""        suppressMessages(library(lme4))        {relevel_cmd}        tryCatch({{            model <- glmer({formula_full}, data=ct_data, family=binomial(link="logit"),                control=glmerControl(optimizer="{GLMM_CONFIG['optimizer']}",                                          optCtrl=list(maxfun={GLMM_CONFIG['maxfun']})),                nAGQ={GLMM_CONFIG['nAGQ']})            cs <- summary(model)$coefficients            list(success=TRUE,                 term_names=rownames(cs),                 estimates=as.numeric(cs[,"Estimate"]),                 std_errors=as.numeric(cs[,"Std. Error"]),                 z_values=as.numeric(cs[,"z value"]),                 p_values=as.numeric(cs[,"Pr(>|z|)"]),                 singular=isSingular(model))        }}, error=function(e) list(success=FALSE, error=as.character(e)))        """        result = ro.r(r_code)        if not result.rx2("success")[0]:            return None        coef_df = pd.DataFrame({            "term":     list(result.rx2("term_names")),            "Estimate": np.array(result.rx2("estimates")),            "SE":       np.array(result.rx2("std_errors")),            "Z":        np.array(result.rx2("z_values")),            "P_value":  np.array(result.rx2("p_values")),        })        singular = bool(result.rx2("singular")[0])        if spec["type"] == "continuous":            primary_terms = [t for t in coef_df["term"] if t == primary]        else:            primary_terms = [t for t in coef_df["term"]                              if t.startswith(primary) and t != primary]        if not primary_terms:            return None        results = []        for term in primary_terms:            row = coef_df[coef_df["term"] == term].iloc[0]            beta, se = float(row["Estimate"]), float(row["SE"])            contrast = primary if spec["type"] == "continuous" else term.replace(primary, "").strip()            results.append({                "Cell_Type": label, "Contrast": contrast,                "N_cells":   n_cells, "N_donors": n_donors,                "SnC_rate":  round(snc_rate, 4),                "Beta":      round(beta, 6), "SE": round(se, 6),                "OR":        round(float(np.exp(beta)), 4),                "CI_low":    round(float(np.exp(beta - 1.96*se)), 4),                "CI_high":   round(float(np.exp(beta + 1.96*se)), 4),                "Z":         round(float(row["Z"]), 4),                "P_value":   round(float(row["P_value"]), 6),                "Singular":  singular,            })        return results    except Exception:        return None# ─────────────────────────────────────────────────────────────────────────────# §8.4.5 — Driver: GLMMs across CTs and specs for one labeling# ─────────────────────────────────────────────────────────────────────────────def run_glmms_for_labels(labels, control_name, scoring_method, replicate_id=None):    out = []    df_temp = df_cells_aligned.copy()    df_temp[OUTCOME_COL_8_4] = labels    df_temp["Donor"] = df_temp["Donor"].astype(str)    for spec in PRIMARY_SPECS:        if spec["var_cell"] not in df_temp.columns:            continue        glmm_cols = ["Donor", "Cell_Type", OUTCOME_COL_8_4, spec["var_cell"]]        for cov in spec["covariates_cell"]:            if cov in df_temp.columns and cov not in glmm_cols:                glmm_cols.append(cov)        df_input = df_temp[glmm_cols].copy().dropna()        df_input[OUTCOME_COL_8_4] = df_input[OUTCOME_COL_8_4].astype(int)        if spec["type"] == "categorical":            df_input[spec["var_cell"]] = df_input[spec["var_cell"]].astype(str)        for cov in spec["covariates_cell"]:            if cov in df_input.columns:                if cov in {"Sex", "Cohort"}:                    df_input[cov] = df_input[cov].astype(str)                else:                    df_input[cov] = pd.to_numeric(df_input[cov], errors="coerce")        df_input = df_input.dropna()        overall = _run_glmm_negctrl(df_input, spec, OUTCOME_COL_8_4, "OVERALL")        if overall:            for r in overall:                r.update({"Control": control_name, "Scoring": scoring_method,                              "Replicate": replicate_id, "Primary": spec["name"]})                out.append(r)        for ct in CELLTYPE_ORDER_PLOT:            ct_data = df_input[df_input["Cell_Type"] == ct].copy()            ct_results = _run_glmm_negctrl(ct_data, spec, OUTCOME_COL_8_4, ct)            if ct_results:                for r in ct_results:                    r.update({"Control": control_name, "Scoring": scoring_method,                                  "Replicate": replicate_id, "Primary": spec["name"]})                    out.append(r)    return out# ─────────────────────────────────────────────────────────────────────────────# §8.4.6 — Run controls# ─────────────────────────────────────────────────────────────────────────────all_glmm_results    = []all_label_summaries = []t0 = time.time()# Control 1: HVGsprint(f"\n{'━'*64}\n  Control 1: HVGs (deterministic, top {N_PANEL_SIZE})\n{'━'*64}")hvg_panel = candidate_hvg[:N_PANEL_SIZE]for sm in CONTROL_SCORING_METHODS:    print(f"\n  ── HVG × {sm} ──")    scores = (_score_genes_84(adata_sub, hvg_panel, "_hvg_score") if sm == "score_genes"              else _log2_median_84(adata_sub, hvg_panel))    labels = _threshold_sd_per_ct(scores, ref_celltypes, SD_MULTIPLIER_8_4)    n_snc = int(labels.sum())    pct_snc = n_snc / n_total * 100    all_label_summaries.append({"Control": "HVG", "Scoring": sm, "Replicate": None,                                  "N_panel_genes": len(hvg_panel),                                  "N_SnC": n_snc, "Pct_SnC": round(pct_snc, 4)})    print(f"    {n_snc:,} SnC ({pct_snc:.2f}%)")    glmm_results = run_glmms_for_labels(labels, "HVG", sm, replicate_id=None)    all_glmm_results.extend(glmm_results)    print(f"    GLMM rows: {len(glmm_results)}")# Control 2: Random genes (replicates)print(f"\n{'━'*64}\n  Control 2: Random genes ({N_REPLICATES} replicates)\n{'━'*64}")for rep in range(N_REPLICATES):    rep_rng = np.random.default_rng(SEED_8_4 + rep)    random_panel = list(rep_rng.choice(candidate_random, size=N_PANEL_SIZE, replace=False))    for sm in CONTROL_SCORING_METHODS:        try:            scores = (_score_genes_84(adata_sub, random_panel, f"_rand_{rep}")                      if sm == "score_genes" else _log2_median_84(adata_sub, random_panel))            labels = _threshold_sd_per_ct(scores, ref_celltypes, SD_MULTIPLIER_8_4)            n_snc = int(labels.sum())            all_label_summaries.append({                "Control": "Random", "Scoring": sm, "Replicate": rep,                "N_panel_genes": len(random_panel), "N_SnC": n_snc,                "Pct_SnC": round(n_snc / n_total * 100, 4)})            all_glmm_results.extend(run_glmms_for_labels(labels, "Random", sm, replicate_id=rep))        except Exception as e:            print(f"    [ERROR] rep {rep} × {sm}: {str(e)[:60]}")    if (rep + 1) % 10 == 0:        print(f"    [{rep+1:3d}/{N_REPLICATES}] elapsed: {(time.time()-t0)/60:.1f} min")# Control 3: Random within HVGs (replicates)print(f"\n{'━'*64}\n  Control 3: Random within HVGs ({N_REPLICATES} replicates)\n{'━'*64}")for rep in range(N_REPLICATES):    rep_rng = np.random.default_rng(SEED_8_4 + 1000 + rep)    rand_hvg_panel = list(rep_rng.choice(candidate_hvg, size=N_PANEL_SIZE, replace=False))    for sm in CONTROL_SCORING_METHODS:        try:            scores = (_score_genes_84(adata_sub, rand_hvg_panel, f"_randhvg_{rep}")                      if sm == "score_genes" else _log2_median_84(adata_sub, rand_hvg_panel))            labels = _threshold_sd_per_ct(scores, ref_celltypes, SD_MULTIPLIER_8_4)            n_snc = int(labels.sum())            all_label_summaries.append({                "Control": "Random_within_HVG", "Scoring": sm, "Replicate": rep,                "N_panel_genes": len(rand_hvg_panel), "N_SnC": n_snc,                "Pct_SnC": round(n_snc / n_total * 100, 4)})            all_glmm_results.extend(run_glmms_for_labels(labels, "Random_within_HVG", sm, replicate_id=rep))        except Exception as e:            print(f"    [ERROR] rep {rep} × {sm}: {str(e)[:60]}")    if (rep + 1) % 10 == 0:        print(f"    [{rep+1:3d}/{N_REPLICATES}] elapsed: {(time.time()-t0)/60:.1f} min")elapsed_total = time.time() - t0print(f"\n  Total compute: {elapsed_total/60:.1f} min")# ─────────────────────────────────────────────────────────────────────────────# §8.4.7 — Save tables (with FDR within group)# ─────────────────────────────────────────────────────────────────────────────df_label_summary = pd.DataFrame(all_label_summaries)df_glmm_negctrl = pd.DataFrame(all_glmm_results)if len(df_glmm_negctrl) > 0:    df_glmm_negctrl["FDR"] = np.nan    df_glmm_negctrl["Significant"] = False    grp_keys = ["Control", "Scoring", "Primary"]    if "Replicate" in df_glmm_negctrl.columns and df_glmm_negctrl["Replicate"].notna().any():        grp_keys.append("Replicate")    for grp_vals, grp_idx in df_glmm_negctrl.groupby(grp_keys, dropna=False).groups.items():        sub = df_glmm_negctrl.loc[grp_idx]        if len(sub) == 0:            continue        df_glmm_negctrl.loc[grp_idx, "FDR"] = bh_correction(sub["P_value"].values)        df_glmm_negctrl.loc[grp_idx, "Significant"] = (            df_glmm_negctrl.loc[grp_idx, "FDR"] < FDR_THRESHOLD)save_table(df_label_summary, "s8_4_negctrl_label_summary")save_table(df_glmm_negctrl,  "s8_4_negctrl_glmm_results")# ─────────────────────────────────────────────────────────────────────────────# §8.4.8 — Replicate-level summary# ─────────────────────────────────────────────────────────────────────────────df_rep_summary = pd.DataFrame()if len(df_glmm_negctrl) > 0:    rep_summary_rows = []    for (control, scoring, primary, ct, contrast), sub in df_glmm_negctrl.groupby(        ["Control", "Scoring", "Primary", "Cell_Type", "Contrast"]):        ors = sub["OR"].dropna().values        if len(ors) == 0:            continue        rep_summary_rows.append({            "Control": control, "Scoring": scoring, "Primary": primary,            "Cell_Type": ct, "Contrast": contrast, "N_replicates": len(sub),            "OR_median":   round(float(np.median(ors)), 4),            "OR_p2.5":     round(float(np.percentile(ors, 2.5)), 4),            "OR_p97.5":    round(float(np.percentile(ors, 97.5)), 4),            "Median_FDR":  round(float(np.nanmedian(sub["FDR"].values)), 4) if sub["FDR"].notna().any() else np.nan,            "Sig_fraction":  round(float(sub["Significant"].mean()), 4),            "Singular_frac": round(float(sub["Singular"].mean()), 4),        })    df_rep_summary = pd.DataFrame(rep_summary_rows)    save_table(df_rep_summary, "s8_4_negctrl_replicate_summary")# ─────────────────────────────────────────────────────────────────────────────# §8.4.9 — Pipeline-bias diagnostic# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*64}\n  PIPELINE BIAS DIAGNOSTIC\n{'─'*64}")print(f"  Question: do random/HVG panels produce significant pathology effects?")print(f"  Expected: NO — sig fraction near α=0.05")print(f"  If sig fraction >> 0.05 → pipeline is biased")if len(df_glmm_negctrl) > 0:    overall_sig = (        df_glmm_negctrl        .groupby(["Control", "Scoring"])        .agg(sig_fraction=("Significant", "mean"),             n_models=("Significant", "count"),             median_OR=("OR", "median"))        .round(4).reset_index()    )    print(f"\n  Overall significance rate by control × scoring:")    print(overall_sig.to_string(index=False))    flagged = overall_sig[overall_sig["sig_fraction"] > 0.10]    if len(flagged) > 0:        print(f"\n  ⚠ Conditions with sig_fraction > 10%:")        print(flagged.to_string(index=False))    else:        print(f"\n  ✓ All conditions ≤ 10% sig — pipeline appears unbiased")# ─────────────────────────────────────────────────────────────────────────────# §8.4.10 — Final summary# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*64}\n  §8.4 SUMMARY\n{'─'*64}")print(f"  Label summaries  : {len(df_label_summary)}")print(f"  GLMM rows        : {len(df_glmm_negctrl)}")if len(df_glmm_negctrl) > 0:    n_reps = df_glmm_negctrl['Replicate'].nunique() if df_glmm_negctrl['Replicate'].notna().any() else 0    print(f"  Unique replicates: {n_reps}")    print(f"  Singular fits    : {df_glmm_negctrl['Singular'].sum()}")print(f"  Total compute    : {elapsed_total/60:.1f} min")print(f"\n✓ §8.4 Negative control complete")print(f"  Tables: s8_4_negctrl_label_summary.csv")print(f"          s8_4_negctrl_glmm_results.csv")print(f"          s8_4_negctrl_replicate_summary.csv")

## Displays — asks 1-4**Why.** Forest and sweep panels for each ask.<sub>source: `03_senescence_burden_model_v2.ipynb` cells 36, 37, 38, 39</sub>

In [ ]:
why("Displays — asks 1-4", "Forest and sweep panels for each ask")

In [ ]:
# ── source: 03_senescence_burden_model_v2.ipynb cell 36 ──# =============================================================================# §9.1 — VISUALIZATIONS FOR §8.1 (Continuous LMM parallel to GLMM)# =============================================================================# Two figures:#   1. GLMM ↔ LMM concordance scatter — log(OR) vs β with quadrant shading.#      Sign-agreement is the headline robustness statistic for Sara's ask #1.#   2. LMM forest plot per spec — β with 95% CI per CT, faceted by contrast.## Both load from §8.1 outputs (s8_1_lmm_results_combined.csv,# s8_1_glmm_lmm_comparison.csv) — robust to kernel restarts.# =============================================================================import matplotlib.pyplot as pltfrom matplotlib.patches import Rectanglefrom matplotlib.lines import Line2Dprint("=" * 64)print(f"§9.1 — VISUALIZATIONS FOR §8.1 (Continuous LMM)")print("=" * 64)def load_or_warn(slug, path_dir=None):    path_dir = path_dir or PATHS["results"]    fpath = os.path.join(path_dir, f"{slug}.csv")    if os.path.exists(fpath):        df = pd.read_csv(fpath)        print(f"  ✓ Loaded {slug}.csv ({len(df):,} rows)")        return df    print(f"  ✗ Missing: {fpath}")    return None# ─────────────────────────────────────────────────────────────────────────────# Load §8.1 outputs# ─────────────────────────────────────────────────────────────────────────────df_lmm     = load_or_warn("s8_1_lmm_results_combined")df_compare = load_or_warn("s8_1_glmm_lmm_comparison")if df_lmm is None or len(df_lmm) == 0:    print(f"\n⊘ §9.1 cannot run without §8.1 outputs")else:    _refresh_celltype_order()    cts_in_data = [ct for ct in CELLTYPE_ORDER_PLOT                       if ct in df_lmm["Cell_Type"].unique()]    # ─────────────────────────────────────────────────────────────────────────    # §9.1.1 — PLOT 1: GLMM ↔ LMM concordance scatter    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  Plot 1: GLMM ↔ LMM concordance\n{'─'*64}")    if df_compare is None or len(df_compare) == 0:        print(f"  ⊘ s8_1_glmm_lmm_comparison.csv missing — skipping concordance plot")    else:        # vs-reference contrasts only (the FDR-corrected ones)        df_c = df_compare[df_compare["Contrast_type"] == "vs_reference"].copy()        df_c = df_c[df_c["log_OR_glmm"].notna() & df_c["Beta_lmm"].notna()].copy()        if len(df_c) == 0:            print(f"  ⊘ No paired GLMM↔LMM rows to plot")        else:            fig1, ax1 = plt.subplots(figsize=(7.5, 7.5), constrained_layout=True)            # Quadrant shading — concordant quadrants light green, discordant light red            xlim = (df_c["log_OR_glmm"].abs().max() * 1.15) if len(df_c) > 0 else 1            ylim = (df_c["Beta_lmm"].abs().max() * 1.15) if len(df_c) > 0 else 1            xlim = max(xlim, 0.1)            ylim = max(ylim, 0.05)            # Concordant quadrants (Q1, Q3) — light green            ax1.add_patch(Rectangle((0, 0), xlim, ylim, facecolor="#e8f5e8",                                       alpha=0.4, zorder=0))            ax1.add_patch(Rectangle((-xlim, -ylim), xlim, ylim, facecolor="#e8f5e8",                                       alpha=0.4, zorder=0))            # Discordant quadrants (Q2, Q4) — light red            ax1.add_patch(Rectangle((-xlim, 0), xlim, ylim, facecolor="#fde8e8",                                       alpha=0.4, zorder=0))            ax1.add_patch(Rectangle((0, -ylim), xlim, ylim, facecolor="#fde8e8",                                       alpha=0.4, zorder=0))            # Origin lines            ax1.axhline(0, color="#444", linewidth=0.7, zorder=1)            ax1.axvline(0, color="#444", linewidth=0.7, zorder=1)            # Diagonal reference line — β ≈ log(OR) under reasonable link conditions            # Slope here is empirical; we draw y=x*0.25 as a heuristic since β is on the            # raw sen_score scale and log(OR) is on log-odds scale (different units).            # Better: draw the OLS regression line for these data points.            if len(df_c) >= 3:                slope = np.polyfit(df_c["log_OR_glmm"], df_c["Beta_lmm"], 1)[0]                xs_diag = np.linspace(-xlim, xlim, 100)                ax1.plot(xs_diag, slope * xs_diag, "--",                            color="#666", linewidth=0.8, alpha=0.7, zorder=1,                            label=f"OLS fit (slope={slope:.3f})")            # Plot points — color by Cell_Type            for ct in cts_in_data + (["OVERALL"] if "OVERALL" in df_c["Cell_Type"].values else []):                sub = df_c[df_c["Cell_Type"] == ct]                if len(sub) == 0:                    continue                color = (LINEAGE_COLORS.get(ct, "#666666") if ct != "OVERALL"                              else "#222222")                marker = "*" if ct == "OVERALL" else "o"                size   = 140 if ct == "OVERALL" else 60                # Stratify by significance: filled = both_sig, hollow = either-only or none                for _, row in sub.iterrows():                    both_sig = bool(row.get("both_sig", False))                    sg       = bool(row.get("Sig_glmm", False))                    sl       = bool(row.get("Sig_lmm", False))                    if both_sig:                        fc, ec, lw = color, "#111", 1.2                    elif sg or sl:                        fc, ec, lw = "white", color, 1.5                    else:                        fc, ec, lw = color, color, 0.4                    ax1.scatter(                        row["log_OR_glmm"], row["Beta_lmm"],                        s=size, marker=marker, facecolor=fc, edgecolor=ec,                        linewidth=lw, alpha=0.92, zorder=4,                    )            # Label points where both_sig=True            for _, row in df_c[df_c["both_sig"] == True].iterrows():                ax1.annotate(                    f"{row['Cell_Type']}\n({row['Contrast']})",                    xy=(row["log_OR_glmm"], row["Beta_lmm"]),                    xytext=(6, 6), textcoords="offset points",                    fontsize=7, color="#222", fontweight="500",                )            # Compute concordance stats            n_total      = len(df_c)            n_concordant = int((np.sign(df_c["log_OR_glmm"]) == np.sign(df_c["Beta_lmm"])).sum())            n_both_sig   = int(df_c["both_sig"].fillna(False).sum())            pct_concord  = n_concordant / n_total * 100 if n_total else 0            # Annotate concordance in upper-left            ax1.text(                0.02, 0.98,                f"Sign concordance: {n_concordant}/{n_total} ({pct_concord:.1f}%)\n"                f"Both significant: {n_both_sig}/{n_total}",                transform=ax1.transAxes,                ha="left", va="top",                fontsize=9, color="#222", fontweight="500",                bbox=dict(facecolor="white", edgecolor="#888",                              alpha=0.92, pad=4, linewidth=0.5),                zorder=5,            )            ax1.set_xlim(-xlim, xlim)            ax1.set_ylim(-ylim, ylim)            ax1.set_xlabel("log(OR)  —  GLMM (binary outcome)", fontsize=10.5)            ax1.set_ylabel("β  —  LMM (continuous sen_score)", fontsize=10.5)            ax1.set_title(                f"GLMM ↔ LMM concordance  ·  per CT × contrast  ·  {DATASET}",                fontsize=11, fontweight="500", pad=8,            )            for spine in ["top", "right"]:                ax1.spines[spine].set_visible(False)            ax1.tick_params(labelsize=9)            # Legend            legend_elems = [                Line2D([0], [0], marker="o", color="w", markerfacecolor="#888",                          markeredgecolor="#111", markersize=8,                          label="Both significant (FDR<0.05)", linewidth=0),                Line2D([0], [0], marker="o", color="w", markerfacecolor="white",                          markeredgecolor="#888", markersize=8,                          label="Either significant", linewidth=0),                Line2D([0], [0], marker="o", color="w", markerfacecolor="#bbb",                          markeredgecolor="#bbb", markersize=8,                          label="Neither significant", linewidth=0),                Line2D([0], [0], marker="*", color="w", markerfacecolor="#222",                          markeredgecolor="#111", markersize=14,                          label="OVERALL (pooled)", linewidth=0),            ]            ax1.legend(handles=legend_elems, loc="lower right", fontsize=8,                          frameon=True, framealpha=0.95)            save_figure(fig1, "s9_1_glmm_lmm_concordance")    # ─────────────────────────────────────────────────────────────────────────    # §9.1.2 — PLOT 2: LMM forest per spec    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  Plot 2: LMM forest plot per spec\n{'─'*64}")    df_lmm_vsref = df_lmm[df_lmm["Contrast_type"] == "vs_reference"].copy()    primary_specs_present = sorted(df_lmm_vsref["Primary"].unique())    for primary in primary_specs_present:        df_p = df_lmm_vsref[df_lmm_vsref["Primary"] == primary].copy()        contrasts_in_p = sorted(df_p["Contrast"].dropna().unique())        if not contrasts_in_p:            continue        # Faceted forest: one column per contrast        n_contrasts = len(contrasts_in_p)        ct_plot = ([ct for ct in cts_in_data                       if ct in df_p["Cell_Type"].unique()]                       + (["OVERALL"] if "OVERALL" in df_p["Cell_Type"].unique() else []))        if not ct_plot:            continue        fig2, axes = plt.subplots(            1, n_contrasts,            figsize=(4.5 * n_contrasts, 0.45 * (len(ct_plot) + 1)),            sharey=True, constrained_layout=True,        )        axes = np.atleast_1d(axes)        # Compute global x-range across contrasts for visual comparability        all_betas    = df_p["Beta"].values        all_ci_lows  = df_p["CI_low"].values        all_ci_highs = df_p["CI_high"].values        x_pad        = (all_ci_highs.max() - all_ci_lows.min()) * 0.08        global_xmin  = float(all_ci_lows.min() - x_pad)        global_xmax  = float(all_ci_highs.max() + x_pad)        y_pos = np.arange(len(ct_plot))[::-1]        for ax, contrast in zip(axes, contrasts_in_p):            df_c = df_p[df_p["Contrast"] == contrast]            ax.axvline(0, color="#888", linewidth=0.7, linestyle="--",                          alpha=0.7, zorder=1)            for k, ct in enumerate(ct_plot):                row = df_c[df_c["Cell_Type"] == ct]                if len(row) == 0:                    continue                beta = float(row["Beta"].iloc[0])                lo   = float(row["CI_low"].iloc[0])                hi   = float(row["CI_high"].iloc[0])                sig  = bool(row["Significant"].iloc[0])                singular = bool(row["Singular"].iloc[0]) if "Singular" in row.columns else False                color = (LINEAGE_COLORS.get(ct, "#666666") if ct != "OVERALL"                              else "#222222")                ax.errorbar(beta, y_pos[k], xerr=[[beta - lo], [hi - beta]],                              fmt="none", ecolor="#888", elinewidth=0.8,                              capsize=2, zorder=2)                marker = "D" if ct == "OVERALL" else "o"                size   = 80 if ct == "OVERALL" else 50                if sig:                    fc, ec, lw = color, "#111", 1.0                else:                    fc, ec, lw = "white", color, 1.2                ax.scatter(beta, y_pos[k], s=size, marker=marker,                              facecolor=fc, edgecolor=ec, linewidth=lw, zorder=3)                # Singular flag                if singular:                    ax.text(global_xmax * 0.98, y_pos[k], "⚠", ha="right", va="center",                              fontsize=10, color="#c0392b", fontweight="700")            ax.set_xlim(global_xmin, global_xmax)            ax.set_yticks(y_pos)            ax.set_yticklabels(ct_plot, fontsize=9)            ax.set_xlabel("β  (LMM, continuous sen_score)", fontsize=9.5)            ax.set_title(f"{contrast}", fontsize=10, fontweight="500")            for spine in ["top", "right"]:                ax.spines[spine].set_visible(False)        fig2.suptitle(            f"LMM β per CT × contrast  ·  {primary}  ·  {DATASET}",            fontsize=11, fontweight="500", y=1.04,        )        slug = f"s9_1_lmm_forest_{primary.lower().replace(' ', '_')}"        save_figure(fig2, slug)    # ─────────────────────────────────────────────────────────────────────────    # §9.1.3 — Final summary    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  §9.1 SUMMARY\n{'─'*64}")    print(f"  Figures saved:")    print(f"    s9_1_glmm_lmm_concordance      (the headline robustness figure)")    for primary in primary_specs_present:        slug = f"s9_1_lmm_forest_{primary.lower().replace(' ', '_')}"        print(f"    {slug}")print(f"\n✓ §9.1 visualizations complete")print(f"  Next: §9.2 threshold sensitivity, §9.3 alt panels, §9.4 negative control")

In [ ]:
# ── source: 03_senescence_burden_model_v2.ipynb cell 37 ──# =============================================================================# §9.2 — VISUALIZATIONS FOR §8.2 (Threshold sensitivity)# =============================================================================# Four figures:#   1. %SnC bar plot per CT × threshold (8 bars per CT: 5 SD + 3 pct)#   2. sen_score distribution per CT with 5 SD thresholds overlaid#      (default 2σ rendered with thicker line)#   3. sen_score distribution per CT with 3 percentile thresholds overlaid#   4. Forest plot per spec — OR vs reference, faceted by threshold## All figures save to PATHS["figures"] in pdf/png/svg.# =============================================================================import colorsysimport matplotlib.pyplot as pltimport matplotlib.gridspec as gridspecfrom matplotlib.lines import Line2Dprint("=" * 64)print(f"§9.2 — VISUALIZATIONS FOR §8.2 (Threshold sensitivity)")print("=" * 64)# ─────────────────────────────────────────────────────────────────────────────# Helpers# ─────────────────────────────────────────────────────────────────────────────def lineage_gradient(base_hex, n=5, lighten_max=0.55):    """Return n shades of base_hex, lightest first, darkest last."""    base_rgb = tuple(int(base_hex[i:i+2], 16) / 255.0 for i in (1, 3, 5))    h, l, s  = colorsys.rgb_to_hls(*base_rgb)    shades = []    for k in range(n):        # k=0: lightest (lighten l to 1-lighten_max), k=n-1: darkest (l unchanged)        l_k = l + (1 - l) * (lighten_max * (1 - k / max(n-1, 1)))        r, g, b = colorsys.hls_to_rgb(h, l_k, s)        shades.append("#%02x%02x%02x" % (int(r*255), int(g*255), int(b*255)))    return shadesdef load_or_warn(slug, path_dir=None):    path_dir = path_dir or PATHS["results"]    fpath = os.path.join(path_dir, f"{slug}.csv")    if os.path.exists(fpath):        df = pd.read_csv(fpath)        print(f"  ✓ Loaded {slug}.csv ({len(df):,} rows)")        return df    print(f"  ✗ Missing: {fpath}")    return None# ─────────────────────────────────────────────────────────────────────────────# §9.2.1 — Load §8.2 outputs# ─────────────────────────────────────────────────────────────────────────────df_thr_combined = load_or_warn("s8_2_threshold_sensitivity_combined")df_pattern      = load_or_warn("s8_2_threshold_pattern_summary")if df_thr_combined is None or len(df_thr_combined) == 0:    print(f"\n⊘ §9.2 cannot run without §8.2 outputs")else:    # Use threshold ordering by kind + value (SD ascending, then pct ascending)    thr_meta = (df_thr_combined[["Threshold_key", "Threshold", "Threshold_kind",                                       "Threshold_value", "Is_default_thr"]]                .drop_duplicates()                .sort_values(["Threshold_kind", "Threshold_value"])                .reset_index(drop=True))    THR_ORDER_SD  = thr_meta[thr_meta["Threshold_kind"] == "sd"]["Threshold_key"].tolist()    THR_ORDER_PCT = thr_meta[thr_meta["Threshold_kind"] == "pct"]["Threshold_key"].tolist()    THR_LABELS    = dict(zip(thr_meta["Threshold_key"], thr_meta["Threshold"]))    THR_VALUES    = dict(zip(thr_meta["Threshold_key"], thr_meta["Threshold_value"]))    THR_KINDS     = dict(zip(thr_meta["Threshold_key"], thr_meta["Threshold_kind"]))    n_sd  = len(THR_ORDER_SD)    n_pct = len(THR_ORDER_PCT)    print(f"\n  Found {n_sd} SD-based + {n_pct} percentile-based thresholds")    cts_in_data = [ct for ct in CELLTYPE_ORDER_PLOT                       if ct in df_thr_combined["Cell_Type"].unique()]    n_cts = len(cts_in_data)    # Defensive harmonization for plot ordering    _refresh_celltype_order()    # ─────────────────────────────────────────────────────────────────────────    # §9.2.2 — PLOT 1: %SnC bar plot per CT × threshold    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  Plot 1: %SnC per CT × threshold\n{'─'*64}")    # Use one row of per-CT counts (any spec row gives the same SnC_rate, since    # SnC labels are spec-agnostic)    snc_rows = (df_thr_combined.groupby(["Cell_Type", "Threshold_key"])                  .agg(SnC_rate=("SnC_rate", "first")).reset_index())    snc_rows = snc_rows[snc_rows["Cell_Type"].isin(cts_in_data)]    fig1, ax1 = plt.subplots(figsize=(11, 5))    bar_w   = 0.9 / (n_sd + n_pct)    x_base  = np.arange(n_cts)    # SD thresholds: lineage_gradient per CT, light → dark for low → high σ    for j, thr_key in enumerate(THR_ORDER_SD):        bar_x = x_base + (j - (n_sd + n_pct - 1) / 2) * bar_w        heights = []        colors  = []        for ct in cts_in_data:            row = snc_rows[(snc_rows["Cell_Type"] == ct) &                              (snc_rows["Threshold_key"] == thr_key)]            heights.append(float(row["SnC_rate"].iloc[0]) * 100 if len(row) else np.nan)            base_col = LINEAGE_COLORS.get(ct, "#666666")            colors.append(lineage_gradient(base_col, n=n_sd)[j])        is_default = thr_meta[thr_meta["Threshold_key"] == thr_key]["Is_default_thr"].iloc[0]        edgecolor = "#222" if is_default else "none"        linewidth = 1.0 if is_default else 0        ax1.bar(bar_x, heights, width=bar_w * 0.95, color=colors,                  edgecolor=edgecolor, linewidth=linewidth,                  label=THR_LABELS[thr_key])    # Percentile thresholds: greyscale gradient    pct_greys = ["#bcbcbc", "#7c7c7c", "#3c3c3c"][:n_pct]    for j, thr_key in enumerate(THR_ORDER_PCT):        bar_x = x_base + (j + n_sd - (n_sd + n_pct - 1) / 2) * bar_w        heights = []        for ct in cts_in_data:            row = snc_rows[(snc_rows["Cell_Type"] == ct) &                              (snc_rows["Threshold_key"] == thr_key)]            heights.append(float(row["SnC_rate"].iloc[0]) * 100 if len(row) else np.nan)        ax1.bar(bar_x, heights, width=bar_w * 0.95,                  color=pct_greys[j], edgecolor="white", linewidth=0.3,                  label=THR_LABELS[thr_key])    ax1.set_xticks(x_base)    ax1.set_xticklabels(cts_in_data, rotation=30, ha="right")    ax1.set_ylabel("%SnC")    ax1.set_title(f"%SnC per CT across {n_sd} SD + {n_pct} percentile thresholds  ·  {DATASET}",                    fontsize=10, fontweight="500")    ax1.spines["top"].set_visible(False)    ax1.spines["right"].set_visible(False)    ax1.legend(ncol=4, fontsize=7, loc="upper right", frameon=False)    save_figure(fig1, "s9_2_pct_snc_per_threshold")    # ─────────────────────────────────────────────────────────────────────────    # §9.2.3 — PLOTS 2 & 3: sen_score distribution histograms    # ─────────────────────────────────────────────────────────────────────────    has_sen_score_data = "df_cells_aligned" in dir() or "df_cells_model" in dir()    if has_sen_score_data and "sen_score" in df_cells_model.columns:        print(f"\n{'─'*64}\n  Plots 2/3: sen_score distribution per CT\n{'─'*64}")        # Pre-extract scores and threshold cutoffs per CT        ct_sen_scores = {}        ct_thresholds_sd  = {}        ct_thresholds_pct = {}        for ct in cts_in_data:            scores = df_cells_model[df_cells_model["Cell_Type"] == ct]["sen_score"].values            if len(scores) == 0:                continue            ct_sen_scores[ct] = scores            mu, sd = float(np.mean(scores)), float(np.std(scores))            ct_thresholds_sd[ct] = {                k: mu + THR_VALUES[k] * sd for k in THR_ORDER_SD            }            ct_thresholds_pct[ct] = {                k: float(np.percentile(scores, 100.0 - THR_VALUES[k]))                for k in THR_ORDER_PCT            }        # Global x-range for visually comparable panels        all_scores  = np.concatenate(list(ct_sen_scores.values()))        global_xmin = float(np.percentile(all_scores, 0.5))        global_xmax = float(np.percentile(all_scores, 99.7))        span        = global_xmax - global_xmin        global_xmin -= span * 0.02        global_xmax += span * 0.05        def _plot_hist_panels(thr_keys, ct_thresholds, fig_title, slug,                                color_dispatch="lineage"):            """Shared rendering: per-CT histograms with threshold lines overlaid."""            n_cols_dist = 3            n_rows_dist = int(np.ceil(n_cts / n_cols_dist))            fig, axes = plt.subplots(                n_rows_dist, n_cols_dist,                figsize=(8.5, 2.5 * n_rows_dist),                constrained_layout=True,            )            axes_flat = np.atleast_1d(axes).flatten()            n_thrs = len(thr_keys)            for panel_idx, ct in enumerate(cts_in_data):                ax = axes_flat[panel_idx]                if ct not in ct_sen_scores:                    ax.set_visible(False)                    continue                scores = ct_sen_scores[ct]                ct_color = LINEAGE_COLORS.get(ct, "#666666")                if color_dispatch == "lineage":                    line_colors = lineage_gradient(ct_color, n=n_thrs)                else:  # greys for percentile                    line_colors = ["#bcbcbc", "#7c7c7c", "#3c3c3c"][:n_thrs]                n, bins, _ = ax.hist(                    scores, bins=60, range=(global_xmin, global_xmax),                    color=ct_color, alpha=0.55, edgecolor="white",                    linewidth=0.3, density=False,                )                ymax_panel = max(n) * 1.32 if len(n) > 0 else 1                thr_vals = ct_thresholds[ct]                for j, thr_key in enumerate(thr_keys):                    cutoff = thr_vals[thr_key]                    is_default = thr_meta[thr_meta["Threshold_key"] == thr_key]["Is_default_thr"].iloc[0]                    line_color = line_colors[j]                    lw         = 2.0 if is_default else 1.2                    alpha      = 1.0 if is_default else 0.85                    ax.axvline(cutoff, color=line_color, linewidth=lw,                                  linestyle="--", alpha=alpha, zorder=3)                    pct_snc = (scores > cutoff).mean() * 100                    label_y_frac = 0.93 - j * (0.85 / max(n_thrs, 1))                    label_text = f"{THR_LABELS[thr_key]}\n{pct_snc:.1f}%"                    ax.text(                        cutoff, ymax_panel * label_y_frac, label_text,                        ha="left" if cutoff < (global_xmin + global_xmax) / 2 else "right",                        va="top",                        fontsize=6.0, color=line_color,                        fontweight=("700" if is_default else "500"),                        bbox=dict(facecolor="white", edgecolor="none",                                      alpha=0.85, pad=1.2),                        zorder=4,                    )                ax.set_xlim(global_xmin, global_xmax)                ax.set_ylim(0, ymax_panel)                ax.set_xlabel("sen_score", fontsize=8)                ax.set_ylabel("Cells", fontsize=8)                ax.tick_params(axis="both", labelsize=7, length=3)                ax.set_title(f"{ct}  (n={len(scores):,})",                                fontsize=9.5, color=ct_color,                                fontweight="500", pad=4)                for spine in ["top", "right"]:                    ax.spines[spine].set_visible(False)                for spine in ["left", "bottom"]:                    ax.spines[spine].set_linewidth(0.6)                    ax.spines[spine].set_color("#333")            for idx in range(n_cts, len(axes_flat)):                axes_flat[idx].set_visible(False)            fig.suptitle(fig_title, fontsize=10, fontweight="500", y=1.01)            save_figure(fig, slug)        # PLOT 2: SD thresholds        _plot_hist_panels(            THR_ORDER_SD, ct_thresholds_sd,            f"sen_score distribution with {n_sd} SD thresholds  ·  {DATASET}  "            f"(default = μ+{SD_DEFAULT_VALUE}σ, thicker line)",            "s9_2_sen_score_distribution_sd",            color_dispatch="lineage",        )        # PLOT 3: percentile thresholds        _plot_hist_panels(            THR_ORDER_PCT, ct_thresholds_pct,            f"sen_score distribution with {n_pct} percentile thresholds  ·  {DATASET}",            "s9_2_sen_score_distribution_pct",            color_dispatch="greys",        )    else:        print(f"\n  ⚠ df_cells_model.sen_score unavailable — skipping histogram plots")    # ─────────────────────────────────────────────────────────────────────────    # §9.2.4 — PLOT 4: Forest plot per spec, faceted by threshold    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  Plot 4: Forest plots per spec across thresholds\n{'─'*64}")    primary_specs_present = sorted(df_thr_combined["Primary"].unique())    for primary in primary_specs_present:        df_p = df_thr_combined[df_thr_combined["Primary"] == primary].copy()        # Headline contrast: first non-reference contrast for this primary        contrasts = sorted(df_p["Contrast"].dropna().unique())        if not contrasts:            continue        focal_contrast = contrasts[0]   # e.g., "AD" if reference is Control        df_focal = df_p[df_p["Contrast"] == focal_contrast].copy()        if len(df_focal) == 0:            continue        # Two-panel forest: left = SD thresholds, right = PCT thresholds        fig4, (axL, axR) = plt.subplots(            1, 2, figsize=(11, 0.55 * (n_cts + 1)),            gridspec_kw={"width_ratios": [n_sd, max(n_pct, 1)]},            sharey=True, constrained_layout=True,        )        # Sort CTs in plot order, plus OVERALL last        ct_plot = ([ct for ct in cts_in_data                       if ct in df_focal["Cell_Type"].unique()]                       + (["OVERALL"] if "OVERALL" in df_focal["Cell_Type"].unique()                              else []))        y_pos = np.arange(len(ct_plot))[::-1]        for ax, thr_keys, kind_label in [            (axL, THR_ORDER_SD,  "SD-based"),            (axR, THR_ORDER_PCT, "Percentile-based"),        ]:            ax.axvline(1, color="#888", linewidth=0.7, linestyle="--", alpha=0.7, zorder=1)            for j, thr_key in enumerate(thr_keys):                df_t = df_focal[df_focal["Threshold_key"] == thr_key]                offset = (j - (len(thr_keys) - 1) / 2) * 0.12                xs, ys, errs_low, errs_high, colors = [], [], [], [], []                for k, ct in enumerate(ct_plot):                    row = df_t[df_t["Cell_Type"] == ct]                    if len(row) == 0:                        continue                    or_val = float(row["OR"].iloc[0])                    lo, hi = float(row["CI_low"].iloc[0]), float(row["CI_high"].iloc[0])                    sig = bool(row["Significant"].iloc[0])                    xs.append(or_val)                    ys.append(y_pos[k] + offset)                    errs_low.append(or_val - lo)                    errs_high.append(hi - or_val)                    if kind_label == "SD-based":                        base_col = LINEAGE_COLORS.get(ct, "#666666")                        col = lineage_gradient(base_col, n=len(thr_keys))[j]                    else:                        col = pct_greys[j] if j < len(pct_greys) else "#3c3c3c"                    colors.append(col)                if not xs:                    continue                # errorbars + dots                ax.errorbar(xs, ys, xerr=[errs_low, errs_high],                              fmt="none", ecolor="#888", elinewidth=0.6,                              capsize=0, zorder=2)                ax.scatter(xs, ys, s=24, c=colors, zorder=3,                              edgecolors="#222", linewidths=0.4)            ax.set_xscale("log")            ax.set_xlabel("OR (log scale)", fontsize=9)            ax.set_yticks(y_pos)            ax.set_yticklabels(ct_plot, fontsize=8.5)            ax.set_title(f"{kind_label} ({len(thr_keys)} thresholds)",                            fontsize=10, fontweight="500")            ax.spines["top"].set_visible(False)            ax.spines["right"].set_visible(False)        fig4.suptitle(            f"OR for {focal_contrast} (vs reference)  ·  {primary}  ·  "            f"per CT × threshold  ·  {DATASET}",            fontsize=10.5, fontweight="500", y=1.04,        )        slug = f"s9_2_forest_{primary.lower().replace(' ', '_')}_{_slugify(focal_contrast)}"        save_figure(fig4, slug)    # ─────────────────────────────────────────────────────────────────────────    # §9.2.5 — Final summary    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  §9.2 SUMMARY\n{'─'*64}")    print(f"  Figures saved:")    print(f"    s9_2_pct_snc_per_threshold")    if has_sen_score_data:        print(f"    s9_2_sen_score_distribution_sd")        print(f"    s9_2_sen_score_distribution_pct")    for primary in primary_specs_present:        contrasts = sorted(df_thr_combined[df_thr_combined["Primary"] == primary]["Contrast"].dropna().unique())        if contrasts:            slug = f"s9_2_forest_{primary.lower().replace(' ', '_')}_{_slugify(contrasts[0])}"            print(f"    {slug}")print(f"\n✓ §9.2 visualizations complete")print(f"  Next: §9.3 alternative scoring (80×80 pairwise heatmaps)")

In [ ]:
# ── source: 03_senescence_burden_model_v2.ipynb cell 38 ──# =============================================================================# §9.3 — VISUALIZATIONS FOR §8.3 (Alternative panels & scoring methods)# =============================================================================# Five figures:#   1. Pairwise Jaccard heatmap  (81×81, hierarchically clustered)#   2. Pairwise Spearman heatmap (81×81, same clustering order)#   3. Summary bar plot — Jaccard with SenePy reference by panel × method#   4. Score distributions per panel (4 methods overlaid per panel = 5 figures)#   5. Per-CT Jaccard heatmap (CTs × configs, vs SenePy reference)# =============================================================================import matplotlib.pyplot as pltimport matplotlib.gridspec as gridspecfrom matplotlib.colors import LinearSegmentedColormapfrom scipy.cluster.hierarchy import linkage, leaves_listfrom scipy.spatial.distance import squareformprint("=" * 64)print(f"§9.3 — VISUALIZATIONS FOR §8.3 (Alternative scoring)")print("=" * 64)def load_or_warn(slug, path_dir=None, indexed=False):    path_dir = path_dir or PATHS["results"]    fpath = os.path.join(path_dir, f"{slug}.csv")    if os.path.exists(fpath):        df = pd.read_csv(fpath, index_col=0 if indexed else None)        print(f"  ✓ Loaded {slug}.csv ({df.shape})")        return df    print(f"  ✗ Missing: {fpath}")    return None# ─────────────────────────────────────────────────────────────────────────────# Load §8.3 outputs# ─────────────────────────────────────────────────────────────────────────────df_per_config = load_or_warn("s8_3_per_config")df_jac_mat    = load_or_warn("s8_3_pairwise_jaccard_matrix",  indexed=True)df_spear_mat  = load_or_warn("s8_3_pairwise_spearman_matrix", indexed=True)df_per_ct     = load_or_warn("s8_3_per_ct_jaccard")if df_per_config is None or len(df_per_config) == 0:    print(f"\n⊘ §9.3 cannot run without §8.3 outputs")else:    _refresh_celltype_order()    # ─────────────────────────────────────────────────────────────────────────    # §9.3.1 — Hierarchical clustering on Jaccard distance    # ─────────────────────────────────────────────────────────────────────────    print(f"\n▸ Hierarchical clustering on 1 - Jaccard")    config_ids = list(df_jac_mat.index)    n_cfg      = len(config_ids)    # Convert similarity to distance, then to condensed form    jac_arr = df_jac_mat.values.astype(float)    jac_arr = np.where(np.isnan(jac_arr), 0, jac_arr)    np.fill_diagonal(jac_arr, 1.0)    dist_arr = 1.0 - jac_arr    dist_arr = (dist_arr + dist_arr.T) / 2  # enforce symmetry    np.fill_diagonal(dist_arr, 0)    try:        Z       = linkage(squareform(dist_arr, checks=False), method="average")        order   = leaves_list(Z)    except Exception as e:        print(f"  ⚠ Clustering failed ({e}); using input order")        order = np.arange(n_cfg)    ordered_ids = [config_ids[i] for i in order]    print(f"  ✓ Cluster order computed for {n_cfg} configs")    # ─────────────────────────────────────────────────────────────────────────    # §9.3.2 — Build panel/method color strips (header annotation)    # ─────────────────────────────────────────────────────────────────────────    PANEL_COLORS = {        "SenePy":          "#005f5f",        "SenMayo":         "#c45a5a",        "Fridman":         "#5a8fc4",        "HernandezSegura": "#c4a05a",        "CellAge":         "#7a5ac4",    }    METHOD_COLORS = {        "score_genes":   "#3c3c3c",        "log2_median":   "#7c7c7c",        "aucell":        "#bcbcbc",        "senepy_native": "#005f5f",        "native":        "#005f5f",    }    THR_COLORS = {        "pooled_top_2pct": "#888",        "pooled_top_5pct": "#bbb",        "per_ct_top_2pct": "#444",        "per_ct_top_5pct": "#222",        "reference":       "#005f5f",    }    def split_cfg(cid):        parts = cid.split("|")        return parts[0], parts[1], parts[2] if len(parts) == 3 else ""    # ─────────────────────────────────────────────────────────────────────────    # §9.3.3 — PLOT 1 & 2: Pairwise heatmaps (Jaccard + Spearman, same order)    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  Plots 1 & 2: Pairwise heatmaps ({n_cfg}×{n_cfg})\n{'─'*64}")    cmap_jac   = "viridis"    cmap_spear = LinearSegmentedColormap.from_list(        "rdbu_smooth", ["#3361a4", "white", "#a43333"]    )    for kind, df_mat, cmap, vrange, slug in [        ("Jaccard",  df_jac_mat,   cmap_jac,   (0, 1),    "s9_3_pairwise_jaccard_heatmap"),        ("Spearman", df_spear_mat, cmap_spear, (-1, 1),   "s9_3_pairwise_spearman_heatmap"),    ]:        if df_mat is None:            print(f"  ⊘ {kind} matrix missing — skipping")            continue        M = df_mat.loc[ordered_ids, ordered_ids].values        fig, ax = plt.subplots(figsize=(11, 11), constrained_layout=True)        gs  = gridspec.GridSpec(            2, 2, width_ratios=[20, 1], height_ratios=[1, 20],            wspace=0.02, hspace=0.02,        )        ax_top  = fig.add_subplot(gs[0, 0])        ax_left = fig.add_subplot(gs[1, 1])  # color strip placement varies        ax_main = fig.add_subplot(gs[1, 0])        im = ax_main.imshow(M, cmap=cmap, vmin=vrange[0], vmax=vrange[1],                              aspect="auto", interpolation="nearest")        ax_main.set_xticks([])        ax_main.set_yticks([])        # Top color strip — panels and methods        strip_h = 2  # rows: panel, method        strip = np.zeros((strip_h, n_cfg, 3))        for j, cid in enumerate(ordered_ids):            panel, method, thr = split_cfg(cid)            pc = PANEL_COLORS.get(panel, "#888")            mc = METHOD_COLORS.get(method, "#888")            strip[0, j] = tuple(int(pc[i:i+2], 16) / 255 for i in (1, 3, 5))            strip[1, j] = tuple(int(mc[i:i+2], 16) / 255 for i in (1, 3, 5))        ax_top.imshow(strip, aspect="auto", interpolation="nearest")        ax_top.set_xticks([])        ax_top.set_yticks([0, 1])        ax_top.set_yticklabels(["Panel", "Method"], fontsize=8)        ax_top.spines["top"].set_visible(False)        ax_top.spines["right"].set_visible(False)        ax_top.spines["bottom"].set_visible(False)        ax_top.spines["left"].set_visible(False)        # Colorbar        cbar = fig.colorbar(im, cax=ax_left, fraction=0.04, shrink=0.6)        cbar.set_label(kind, fontsize=10)        fig.suptitle(            f"{kind} — pairwise across {n_cfg} configurations  ·  "            f"hierarchically clustered (avg-linkage on 1−Jaccard)  ·  {DATASET}",            fontsize=10.5, fontweight="500", y=1.005,        )        # Build legend for color strips        from matplotlib.patches import Patch        legend_panels = [Patch(color=c, label=p) for p, c in PANEL_COLORS.items()]        legend_methods = [Patch(color=c, label=m) for m, c in METHOD_COLORS.items()                              if m != "native"]        fig.legend(handles=legend_panels + legend_methods,                      loc="lower center", ncol=5, fontsize=7.5,                      bbox_to_anchor=(0.5, -0.02), frameon=False)        save_figure(fig, slug)        print(f"  ✓ {slug}")    # ─────────────────────────────────────────────────────────────────────────    # §9.3.4 — PLOT 3: Summary bar plot — Jaccard with SenePy by panel × method    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  Plot 3: Median Jaccard with SenePy reference\n{'─'*64}")    df_summary = (df_per_config[df_per_config["config_id"] != "SenePy|native|reference"]                       .groupby(["Panel", "Method"])                       .agg(Jaccard_median=("Jaccard_with_ref", "median"),                            Jaccard_min=("Jaccard_with_ref", "min"),                            Jaccard_max=("Jaccard_with_ref", "max"))                       .reset_index())    panels_order  = list(ALT_PANELS_8_3.keys())    methods_order = SCORING_METHODS_8_3    fig3, ax3 = plt.subplots(figsize=(10, 5))    bar_w = 0.18    x_base = np.arange(len(panels_order))    for j, method in enumerate(methods_order):        sub = df_summary[df_summary["Method"] == method]        heights, errs_low, errs_high = [], [], []        for p in panels_order:            row = sub[sub["Panel"] == p]            if len(row) == 0:                heights.append(np.nan); errs_low.append(0); errs_high.append(0)                continue            med = float(row["Jaccard_median"].iloc[0])            mn  = float(row["Jaccard_min"].iloc[0])            mx  = float(row["Jaccard_max"].iloc[0])            heights.append(med)            errs_low.append(med - mn)            errs_high.append(mx - med)        ax3.bar(x_base + (j - 1.5) * bar_w, heights, width=bar_w * 0.92,                  color=METHOD_COLORS.get(method, "#888"),                  yerr=[errs_low, errs_high], capsize=2,                  error_kw={"elinewidth": 0.6, "ecolor": "#666"},                  label=method, edgecolor="white", linewidth=0.5)    ax3.set_xticks(x_base)    ax3.set_xticklabels(panels_order, rotation=20, ha="right")    ax3.set_ylabel("Jaccard with SenePy reference (median ± min-max)", fontsize=10)    ax3.set_ylim(0, 1)    ax3.axhline(0.5, color="#888", linewidth=0.5, linestyle=":", alpha=0.7)    ax3.set_title(        f"Agreement of alt configurations with SenePy native reference  ·  {DATASET}",        fontsize=10.5, fontweight="500",    )    ax3.spines["top"].set_visible(False)    ax3.spines["right"].set_visible(False)    ax3.legend(ncol=2, fontsize=8, loc="upper right", frameon=False, title="Method")    save_figure(fig3, "s9_3_summary_jaccard_bar")    # ─────────────────────────────────────────────────────────────────────────    # §9.3.5 — PLOT 4: Per-panel score distributions (4 methods overlaid)    # ─────────────────────────────────────────────────────────────────────────    score_mat_path_parquet = os.path.join(PATHS["results"], "s8_3_score_matrix.parquet")    score_mat_path_csv     = os.path.join(PATHS["results"], "s8_3_score_matrix.csv")    df_score_mat = None    if os.path.exists(score_mat_path_parquet):        df_score_mat = pd.read_parquet(score_mat_path_parquet)        print(f"  ✓ Loaded score_matrix.parquet ({df_score_mat.shape})")    elif os.path.exists(score_mat_path_csv):        df_score_mat = pd.read_csv(score_mat_path_csv, index_col=0)        print(f"  ✓ Loaded score_matrix.csv ({df_score_mat.shape})")    if df_score_mat is not None:        print(f"\n{'─'*64}\n  Plot 4: Per-panel score distributions\n{'─'*64}")        for panel in panels_order:            # Find columns for this panel (excluding the SenePy reference)            panel_cols = [c for c in df_score_mat.columns                              if c.startswith(f"{panel}|") and not c.endswith("|reference")]            if not panel_cols:                continue            fig4, ax4 = plt.subplots(figsize=(7, 4))            for j, method in enumerate(methods_order):                method_cols = [c for c in panel_cols if c.split("|")[1] == method]                if not method_cols:                    continue                # Use the 1st threshold's score column (scores are threshold-independent)                col = method_cols[0]                vals = df_score_mat[col].dropna().values                if len(vals) == 0:                    continue                # Z-normalize for visual comparability (each method has its own scale)                vals_z = (vals - np.mean(vals)) / max(np.std(vals), 1e-6)                ax4.hist(vals_z, bins=80, alpha=0.45,                            color=METHOD_COLORS.get(method, "#888"),                            label=method, density=True, edgecolor="none")            ax4.set_xlabel("Score (z-normalized within method)", fontsize=10)            ax4.set_ylabel("Density", fontsize=10)            ax4.set_xlim(-3, 6)            ax4.set_title(f"Score distribution — {panel} panel  ·  {DATASET}",                            fontsize=10, fontweight="500")            ax4.spines["top"].set_visible(False)            ax4.spines["right"].set_visible(False)            ax4.legend(fontsize=8, frameon=False)            save_figure(fig4, f"s9_3_score_distribution_{panel.lower()}")    else:        print(f"  ⊘ Score matrix not found — skipping per-panel distribution figures")    # ─────────────────────────────────────────────────────────────────────────    # §9.3.6 — PLOT 5: Per-CT Jaccard heatmap (CTs × configs)    # ─────────────────────────────────────────────────────────────────────────    if df_per_ct is not None and len(df_per_ct) > 0:        print(f"\n{'─'*64}\n  Plot 5: Per-CT Jaccard heatmap\n{'─'*64}")        cts_in_data = [ct for ct in CELLTYPE_ORDER_PLOT                          if ct in df_per_ct["Cell_Type"].unique()]        configs_in_data = [c for c in ordered_ids if c != "SenePy|native|reference"                              and c in df_per_ct["config_id"].values]        if cts_in_data and configs_in_data:            pivot = (df_per_ct.pivot(index="Cell_Type", columns="config_id",                                          values="Jaccard")                          .reindex(index=cts_in_data, columns=configs_in_data))            fig5, ax5 = plt.subplots(                figsize=(0.18 * len(configs_in_data) + 4,                         0.45 * len(cts_in_data) + 1.5),                constrained_layout=True,            )            im = ax5.imshow(pivot.values, cmap="viridis", vmin=0, vmax=1,                              aspect="auto", interpolation="nearest")            ax5.set_yticks(range(len(cts_in_data)))            ax5.set_yticklabels(cts_in_data, fontsize=9)            ax5.set_xticks([])            cbar5 = fig5.colorbar(im, ax=ax5, fraction=0.025, shrink=0.7)            cbar5.set_label("Jaccard vs SenePy reference", fontsize=9)            ax5.set_title(                f"Per-CT Jaccard with SenePy reference across {len(configs_in_data)} configs  ·  {DATASET}",                fontsize=10, fontweight="500",            )            save_figure(fig5, "s9_3_per_ct_jaccard_heatmap")            print(f"  ✓ s9_3_per_ct_jaccard_heatmap")    # ─────────────────────────────────────────────────────────────────────────    # §9.3.7 — Summary    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  §9.3 SUMMARY\n{'─'*64}")    print(f"  Figures saved:")    print(f"    s9_3_pairwise_jaccard_heatmap      ({n_cfg}×{n_cfg})")    print(f"    s9_3_pairwise_spearman_heatmap     ({n_cfg}×{n_cfg})")    print(f"    s9_3_summary_jaccard_bar           (median Jaccard by panel × method)")    print(f"    s9_3_score_distribution_<panel>    (per panel score densities)")    print(f"    s9_3_per_ct_jaccard_heatmap        (CT × config)")print(f"\n✓ §9.3 visualizations complete")print(f"  Next: §9.4 negative control")

In [ ]:
# ── source: 03_senescence_burden_model_v2.ipynb cell 39 ──# =============================================================================# §9.4 — VISUALIZATIONS FOR §8.4 (Negative control with random panels)# =============================================================================# Three figures:#   1. Sig fraction barplot     — control × scoring (pipeline-bias diagnostic)#   2. Per-CT FPR heatmap       — CT × control × scoring with α=5% reference#   3. Null vs SenePy           — per validation CT, OR distribution under null# =============================================================================import matplotlib.pyplot as pltfrom matplotlib.patches import Rectanglefrom matplotlib.lines import Line2Dprint("=" * 64)print(f"§9.4 — VISUALIZATIONS FOR §8.4 (Negative control)")print("=" * 64)def load_or_warn(slug, path_dir=None):    path_dir = path_dir or PATHS["results"]    fpath = os.path.join(path_dir, f"{slug}.csv")    if os.path.exists(fpath):        df = pd.read_csv(fpath)        print(f"  ✓ Loaded {slug}.csv ({len(df):,} rows)")        return df    print(f"  ✗ Missing: {fpath}")    return None# ─────────────────────────────────────────────────────────────────────────────# Load §8.4 outputs# ─────────────────────────────────────────────────────────────────────────────df_glmm_neg = load_or_warn("s8_4_negctrl_glmm_results")df_rep_summ = load_or_warn("s8_4_negctrl_replicate_summary")if df_glmm_neg is None or len(df_glmm_neg) == 0:    print(f"\n⊘ §9.4 cannot run without §8.4 outputs")else:    _refresh_celltype_order()    CONTROL_ORDER = ["HVG", "Random", "Random_within_HVG"]    CONTROL_COLORS = {        "HVG":               "#5a8fc4",        "Random":            "#7c7c7c",        "Random_within_HVG": "#a05ac4",    }    SCORING_ORDER = ["score_genes", "log2_median"]    cts_in_data = [ct for ct in CELLTYPE_ORDER_PLOT                       if ct in df_glmm_neg["Cell_Type"].unique()]    # ─────────────────────────────────────────────────────────────────────────    # §9.4.1 — PLOT 1: Sig fraction barplot    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  Plot 1: Sig fraction barplot\n{'─'*64}")    sig_df = (df_glmm_neg              .groupby(["Control", "Scoring", "Primary"])              .agg(sig_fraction=("Significant", "mean"),                   n_models=("Significant", "count"))              .reset_index())    primaries = sorted(sig_df["Primary"].unique())    n_primaries = len(primaries)    fig1, axes1 = plt.subplots(1, n_primaries,                                       figsize=(5 * n_primaries, 4),                                       sharey=True, constrained_layout=True)    axes1 = np.atleast_1d(axes1)    for ax, primary in zip(axes1, primaries):        sub = sig_df[sig_df["Primary"] == primary]        x_base = np.arange(len(SCORING_ORDER))        bar_w  = 0.25        for j, ctrl in enumerate(CONTROL_ORDER):            heights = []            for sm in SCORING_ORDER:                row = sub[(sub["Control"] == ctrl) & (sub["Scoring"] == sm)]                heights.append(float(row["sig_fraction"].iloc[0]) if len(row) else 0)            ax.bar(x_base + (j - 1) * bar_w, heights, width=bar_w * 0.92,                      color=CONTROL_COLORS[ctrl], label=ctrl,                      edgecolor="white", linewidth=0.5)        # α=0.05 reference line        ax.axhline(0.05, color="#c0392b", linewidth=1.0, linestyle="--",                       alpha=0.85, zorder=1, label="α = 0.05")        ax.set_xticks(x_base)        ax.set_xticklabels(SCORING_ORDER, fontsize=9)        ax.set_ylabel("Significant fraction", fontsize=9.5)        ax.set_title(f"Primary: {primary}", fontsize=10, fontweight="500")        ax.set_ylim(0, max(0.15, sig_df["sig_fraction"].max() * 1.2))        ax.spines["top"].set_visible(False)        ax.spines["right"].set_visible(False)    axes1[0].legend(fontsize=7.5, frameon=False, loc="upper left")    fig1.suptitle(        f"Pipeline-bias diagnostic — sig fraction across negative controls  ·  {DATASET}",        fontsize=10.5, fontweight="500", y=1.02,    )    save_figure(fig1, "s9_4_sig_fraction_barplot")    # ─────────────────────────────────────────────────────────────────────────    # §9.4.2 — PLOT 2: Per-CT FPR heatmap    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  Plot 2: Per-CT FPR heatmap\n{'─'*64}")    primaries_h = sorted(df_glmm_neg["Primary"].unique())    n_specs_h   = len(primaries_h)    n_cts       = len(cts_in_data)    fig2, axes2 = plt.subplots(        1, n_specs_h, figsize=(8.5, 0.5 * n_cts + 1.8),        sharey=True, constrained_layout=True,    )    axes2 = np.atleast_1d(axes2)    # Each cell: %FDR-sig out of replicate models for (CT, control, scoring)    fpr_columns = []   # control × scoring labels (column blocks)    for ctrl in CONTROL_ORDER:        for sm in SCORING_ORDER:            fpr_columns.append((ctrl, sm))    im_last = None    for ax, primary in zip(axes2, primaries_h):        df_p = df_glmm_neg[df_glmm_neg["Primary"] == primary]        M = np.full((n_cts, len(fpr_columns)), np.nan)        for ci, (ctrl, sm) in enumerate(fpr_columns):            sub = df_p[(df_p["Control"] == ctrl) & (df_p["Scoring"] == sm)]            for ri, ct in enumerate(cts_in_data):                ct_sub = sub[sub["Cell_Type"] == ct]                if len(ct_sub) == 0:                    continue                M[ri, ci] = float(ct_sub["Significant"].mean())        im = ax.imshow(M, cmap="Reds", vmin=0, vmax=0.20,                          aspect="auto", interpolation="nearest")        im_last = im        # Mark cells exceeding α=5% with a red box        for ri in range(n_cts):            for ci in range(len(fpr_columns)):                if not np.isnan(M[ri, ci]) and M[ri, ci] > 0.05:                    rect = Rectangle((ci - 0.45, ri - 0.45), 0.9, 0.9,                                          fill=False, edgecolor="#c0392b",                                          linewidth=1.5, zorder=3)                    ax.add_patch(rect)                # Annotate value                if not np.isnan(M[ri, ci]):                    ax.text(ci, ri, f"{M[ri, ci]:.2f}",                              ha="center", va="center", fontsize=7,                              color="#222" if M[ri, ci] < 0.10 else "white")        ax.set_yticks(range(n_cts))        ax.set_yticklabels(cts_in_data, fontsize=8.5)        ax.set_xticks(range(len(fpr_columns)))        ax.set_xticklabels([f"{c}\n{s}" for c, s in fpr_columns],                              fontsize=7, rotation=45, ha="right")        ax.set_title(f"{primary}", fontsize=10, fontweight="500")    cbar2 = fig2.colorbar(im_last, ax=axes2, fraction=0.025, pad=0.01, shrink=0.85)    cbar2.set_label("Fraction of replicates with FDR<0.05", fontsize=8)    fig2.suptitle(        f"Per-CT false positive rate (red box = above α=5%)  ·  {DATASET}",        fontsize=10.5, fontweight="500", y=1.04,    )    save_figure(fig2, "s9_4_per_ct_fpr_heatmap")    # ─────────────────────────────────────────────────────────────────────────    # §9.4.3 — PLOT 3: Null OR distribution per validation CT    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  Plot 3: Null vs reference OR per CT\n{'─'*64}")    # Get reference OR from §5 GLMM (if available) for each spec    ref_glmm_or = {}   # (primary, ct, contrast) → OR    if "df_glmm" in dir() and len(df_glmm) > 0:        for _, row in df_glmm[df_glmm["Contrast_type"] == "vs_reference"].iterrows():            ref_glmm_or[(row["Primary"], row["Cell_Type"], row["Contrast"])] = float(row["OR"])    # CTs to focus on: MANUAL_VALIDATION_CTS or top GLMM-sig CTs    if "MANUAL_VALIDATION_CTS" in dir():        focus_cts = [ct for ct in MANUAL_VALIDATION_CTS if ct in cts_in_data]    else:        focus_cts = cts_in_data[:4]    print(f"  Focus CTs: {focus_cts}")    for focus_ct in focus_cts:        df_p = df_glmm_neg[(df_glmm_neg["Cell_Type"] == focus_ct)                              & df_glmm_neg["Replicate"].notna()]        if len(df_p) == 0:            continue        primaries_f = sorted(df_p["Primary"].unique())        if not primaries_f:            continue        n_rows = len(SCORING_ORDER)        n_cols = len(primaries_f)        fig3, axes3 = plt.subplots(            n_rows, n_cols,            figsize=(4.0 * n_cols, 2.8 * n_rows),            sharex=False, constrained_layout=True,        )        axes3 = np.atleast_2d(axes3)        for ri, sm in enumerate(SCORING_ORDER):            for ci, primary in enumerate(primaries_f):                ax = axes3[ri, ci]                df_cell = df_p[(df_p["Scoring"] == sm) & (df_p["Primary"] == primary)]                if len(df_cell) == 0:                    ax.set_visible(False)                    continue                # Show null OR distribution (random + random_within_HVG only)                # HVG is deterministic, so it's a single line                for ctrl in CONTROL_ORDER:                    sub = df_cell[df_cell["Control"] == ctrl]                    if len(sub) == 0:                        continue                    ors = sub["OR"].dropna().values                    if len(ors) == 0:                        continue                    if ctrl == "HVG":                        # HVG is deterministic — single OR per (CT, contrast); show as line                        for or_val in np.unique(ors):                            ax.axvline(or_val, color=CONTROL_COLORS[ctrl],                                          linewidth=2.0, linestyle="-",                                          alpha=0.9, label=f"{ctrl} (deterministic)",                                          zorder=3)                    else:                        ax.hist(ors, bins=24, alpha=0.55,                                  color=CONTROL_COLORS[ctrl], label=f"{ctrl} (n={len(ors)})",                                  edgecolor="white", linewidth=0.3, density=True,                                  zorder=2)                # OR=1 reference                ax.axvline(1, color="#444", linewidth=0.8, linestyle="--",                              alpha=0.7, zorder=1)                # Reference SenePy OR (one value per contrast)                contrasts_in_cell = sorted(df_cell["Contrast"].unique())                for contrast in contrasts_in_cell:                    ref_key = (primary, focus_ct, contrast)                    if ref_key in ref_glmm_or:                        ref_or = ref_glmm_or[ref_key]                        ax.axvline(ref_or, color="#005f5f", linewidth=2.0,                                      linestyle="-",                                      alpha=0.95, zorder=4)                        ax.text(ref_or, ax.get_ylim()[1] * 0.95,                                  f" SenePy {contrast}\n OR={ref_or:.2f}",                                  ha="left", va="top", fontsize=7.5,                                  color="#005f5f", fontweight="500")                ax.set_xscale("log")                ax.set_xlabel("OR (log scale)", fontsize=9)                if ci == 0:                    ax.set_ylabel(f"{sm}\nDensity", fontsize=9)                ax.set_title(f"{primary}", fontsize=9.5, fontweight="500")                ax.spines["top"].set_visible(False)                ax.spines["right"].set_visible(False)        # Single legend at bottom        legend_elements = [            Line2D([0], [0], color=CONTROL_COLORS["HVG"], linewidth=2,                       label="HVG (deterministic)"),            Rectangle((0, 0), 1, 1, color=CONTROL_COLORS["Random"],                          alpha=0.55, label="Random (null distribution)"),            Rectangle((0, 0), 1, 1, color=CONTROL_COLORS["Random_within_HVG"],                          alpha=0.55, label="Random within HVG (null)"),            Line2D([0], [0], color="#005f5f", linewidth=2,                       label="SenePy reference (§5 GLMM)"),        ]        fig3.legend(handles=legend_elements, loc="lower center", ncol=4,                       fontsize=8, bbox_to_anchor=(0.5, -0.03), frameon=False)        fig3.suptitle(            f"Null OR distribution vs SenePy reference  ·  {focus_ct}  ·  {DATASET}",            fontsize=10.5, fontweight="500", y=1.02,        )        save_figure(fig3, f"s9_4_null_vs_senepy_{focus_ct.lower()}")        print(f"  ✓ s9_4_null_vs_senepy_{focus_ct.lower()}")    # ─────────────────────────────────────────────────────────────────────────    # §9.4.4 — Summary    # ─────────────────────────────────────────────────────────────────────────    print(f"\n{'─'*64}\n  §9.4 SUMMARY\n{'─'*64}")    print(f"  Figures saved:")    print(f"    s9_4_sig_fraction_barplot        (pipeline-bias diagnostic)")    print(f"    s9_4_per_ct_fpr_heatmap          (CT × control × scoring)")    print(f"    s9_4_null_vs_senepy_<ct>         (one figure per focus CT)")    # Final read on bias    overall_sig = (df_glmm_neg.groupby(["Control", "Scoring"])                       .agg(sig_fraction=("Significant", "mean"))                       .reset_index())    flagged = overall_sig[overall_sig["sig_fraction"] > 0.10]    if len(flagged) > 0:        print(f"\n  ⚠ Conditions with sig_fraction > 10%:")        print(flagged.to_string(index=False))    else:        print(f"\n  ✓ All conditions ≤ 10% sig — pipeline appears unbiased")print(f"\n✓ §9.4 visualizations complete")print(f"  Module 03 v2 visualization batch DONE.")

## Ask 5 — depth-matched downsampling**Why.** ⟨NEW — NOT IMPLEMENTED⟩ Downsample UMIs to a common depth and re-score. If the effect survives at matched depth it is biology, not a depth artifact. This is the strongest defense of the decision to report the depth coupling rather than regress it out — and unlike a second regression it removes the confound by design rather than by model. No source exists for this; it must be written.

In [ ]:
raise NotImplementedError(    "Ask 5 (depth-matched downsampling) has no source implementation. "    "Write it or remove the claim from the README validation table.")

## GATE**Why.** A failed robustness ask does not invalidate the pipeline, but it changes what may be claimed. Modules 05 onward are interpretable only in light of what happened here — so the outcome is recorded explicitly rather than left implicit.

In [ ]:
# ⟨NEW⟩ exit gate — record, do not silently passwhy("04 GATE", "Which robustness asks passed, and what may be claimed downstream?")print("  Record per ask: direction stable / attenuated / reversed.")print("  Instability is reportable, not fatal — but it must be reported, not buried.")print("  Ask 5 is NOT IMPLEMENTED; the README claim is unsupported until it is.")